In [ ]:
import torch
import torch.nn as nn
from torch.nn.parameter import Parameter
import torch.nn.functional as F

#The ANFIS and other necesary functions

##Functions

###MFs

In [ ]:
gaussian2_params = ['mu', 'sigma']
def gaussian2(x, p):
    return torch.exp(torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
gaussian3_params = ['mu', 'sigma', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [ ]:
bell_params = ["a", "b", "c"]
def bell(x, p):
    return 1 / (1 + torch.pow(torch.abs((x - p[:, :, 2]) / p[:, :, 0]), 2 * p[:, :, 1]))

In [ ]:
triangular_params = ["a", "b"]
def triangular(x, p):
    return torch.clamp(1 - torch.abs((x - p[:, :, 1]) / p[:, :, 0]), min=0)

In [ ]:
trapezoidal_params = ["a", "b", "c", "d"]
def trapezoidal(x, p):
    return torch.min(torch.clamp((x - p[:, :, 0]) / (p[:, :, 1] - p[:, :, 0]), min=0, max=1), torch.clamp((p[:, :, 3] - x) / (p[:, :, 3] - p[:, :, 2]), min=0, max=1))

###Consequent Functions

In [ ]:
def weighted_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1]).mul(w)

###Other Functions

In [ ]:
def sum(x):
    return torch.sum(x, dim=-1)

##Layers

In [ ]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly

    Other attributes:
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, init_rules=3, mf=gaussian2, params=['mu', 'sigma'], x_train=[]):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.init_rules = init_rules
        self.mf = mf
        self.params = params
        if x_train == []:
            self.premises = Parameter(torch.rand(input_size, init_rules, len(params)), requires_grad=True)
        else:
            self.premises = Parameter(self.init_premises(x_train), requires_grad=True)

    def init_premises(self, x_train):
        premises = torch.zeros(self.input_size, self.init_rules, len(self.params))
        if self.init_rules > 1:
            min = torch.min(x_train, dim=0).values
            max = torch.max(x_train, dim=0).values
            stp = (max - min) / (self.init_rules - 1)
            for i in range(self.input_size):
                h = torch.arange(min[i], max[i] + stp[i], stp[i])
                premises[i, :, 0] = h[:self.init_rules]
                premises[i, :, 1] = stp[i]/2
                if (len(self.params) == 3): premises[i, :, 2] = 2
        else:
            for i in range(self.input_size):
                premises[i, :, 0] = torch.mean(x_train[:, i])
                premises[i, :, 1] = torch.std(x_train[:, i])
                if (len(self.params) == 3): premises[i, :, 2] = 2
        return premises

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.init_rules):
                print(f"        rule {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]

In [ ]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [ ]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of init_rules
        function : function used as a consequent function

    Other attributes:
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, init_rules=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.init_rules = init_rules
        self.input_size = input_size
        self.function = function
        self.consequents = Parameter(torch.rand(init_rules, input_size + 1), requires_grad=True)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.init_rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [ ]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

##Strucuture

In [ ]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters namesr
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, input_size, init_rules=3, cf=weighted_linear, mf=gaussian2, of=sum, mf_params=['mu', 'sigma'], x_train=[]):
        super(Type3ANFIS, self).__init__()
        self.input_size = input_size
        self.mf_params = mf_params
        if x_train == []:
            self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params)
        else:
            self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params, x_train)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, init_rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(output)
        return output

    def firing_levels(self, x):
        x = self.fuzzify_layer(x)
        fl = x.prod(dim=x.dim()-2)
        return fl

    def outputs_by_rule(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        return output

    @property
    def rules(self):
        return self.consequents.shape[0]

    @property
    def premises_structure(self):
        self.fuzzify_layer.premises_structure

    @property
    def premises(self):
        return self.fuzzify_layer.premises

    def set_premises(self, premises):
        self.fuzzify_layer.premises = premises

    @property
    def consequents_structure(self):
        self.consequent_layer.consequents_structure

    @property
    def consequents(self):
        return self.consequent_layer.consequents

    def set_consequents(self, consequents):
        self.consequent_layer.consequents = consequents

#Parameters testing

##DataLoader

In [ ]:
import torch.utils.data as data

In [ ]:
x_train = torch.rand(100, 2)*10

In [ ]:
def function(tensor):
    results = torch.zeros(tensor.size(0))

    for i in range(tensor.size(0)):
        x0 = tensor[i, 0].item()
        x1 = tensor[i, 1].item()

        if x0 > x1:
            if int(x0) % 2 == 0:
                if int(x1) % 2 == 0:
                    results[i] = x0 * x1 / 4
                else:
                    results[i] = x0 * x1
            else:
                if int(x1) % 2 == 0:
                    results[i] = (x0 - x1) * 3
                else:
                    results[i] = (x0 - x1) * 2
        else:
            if int(x0) % 2 == 0:
                if int(x1) % 2 == 0:
                    results[i] = (x0 + x1) / 2
                else:
                    results[i] = (x0 + x1)
            else:
                if int(x1) % 2 == 0:
                    results[i] = (x1 - x0 + 2) * 2
                else:
                    results[i] = (x1 - x0 + 2)

    return results.unsqueeze(1)

In [ ]:
y_train = function(x_train)
y_train.requires_grad

False

In [ ]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 8, shuffle = True)

In [ ]:
data_iter = iter(loader)
batch1 = next(data_iter)
x_batch1, y_batch1 = batch1
x_batch1

tensor([[9.9986, 7.8104],
        [5.0889, 8.1364],
        [1.8265, 2.4428],
        [4.0985, 0.6733],
        [1.7619, 1.6569],
        [0.9139, 2.8287],
        [6.3156, 4.6664],
        [8.2050, 3.2290]])

## Gradient testing

In [ ]:
model = Type3ANFIS(x_train.shape[1], x_train=x_train)

In [ ]:
print(model.premises)
print(model.premises.grad)
#print(model.premises.grad_fn)
print(model.consequents)
print(model.consequents.grad)

Parameter containing:
tensor([[[0.1112, 2.4719],
         [5.0549, 2.4719],
         [9.9986, 2.4719]],

        [[0.0238, 2.4914],
         [5.0065, 2.4914],
         [9.9892, 2.4914]]], requires_grad=True)
None
Parameter containing:
tensor([[0.7470, 0.0546, 0.0371],
        [0.5600, 0.8793, 0.3308],
        [0.3740, 0.7422, 0.9564]], requires_grad=True)
None


In [ ]:
trainable_parameters = [model.premises, model.consequents]

optimizer = torch.optim.SGD(trainable_parameters, lr=0.001)
prediction = model(x_batch1)
prediction.retain_grad()
prediction.requires_grad_()
loss = (y_batch1 - prediction).pow(2).sum()
print(loss)

tensor(4669.5156, grad_fn=<SumBackward0>)


In [ ]:
loss.backward()

In [ ]:
print(prediction)
print(prediction.grad)
#print(prediction.grad_fn)

tensor([7.9325, 4.3199, 3.4522, 2.9889, 2.8449, 3.3974, 5.3529, 6.3574],
       grad_fn=<SumBackward1>)
tensor([ 14.2453, -43.5560, -57.4382, -64.8513, -67.1551, -58.3149, -27.0284,
        -10.9554])


In [ ]:
optimizer.step()
print(model.premises)
print(model.premises.grad)
print(model.consequents)
print(model.consequents.grad)

Parameter containing:
tensor([[[ 0.1203,  2.4943],
         [ 5.0548,  2.4718],
         [10.0039,  2.4635]],

        [[ 0.0316,  2.5088],
         [ 5.0064,  2.4911],
         [ 9.9957,  2.4780]]], requires_grad=True)
tensor([[[ -9.1483, -22.4515],
         [  0.1451,   0.1026],
         [ -5.2645,   8.3627]],

        [[ -7.8099, -17.4233],
         [  0.1348,   0.2617],
         [ -6.4808,  13.3820]]])
Parameter containing:
tensor([[1.0438, 0.4292, 0.0978],
        [0.5621, 0.8809, 0.3311],
        [0.9571, 1.2308, 1.2103]], requires_grad=True)
tensor([[-2.9686e+02, -3.7456e+02, -6.0771e+01],
        [-2.0996e+00, -1.6449e+00, -3.4999e-01],
        [-5.8316e+02, -4.8863e+02, -2.5393e+02]])


Both premises and consequents are updated, so its possible to start experimenting with learning algorithms.


#Learning Algorithms

##Ordinary Least-Square (OLS)

###Proposal 1
A rustic method for the OLS algorithm that operates on the entire dataset for updating the consequent parameters and uses a loop trought the training dataset for updating the premises parameters.

####Consequents

In [ ]:
x = torch.tensor([[5., 5., 1.],
                  [6., 5., 2.],
                  [7., 3., 4.],
                  [1., 3., 5.],
                  [4., 5., 9.],
                  [4., 6., 1.],
                  [1., 2., 3.],
                  [6., 8., 3.]])
y = torch.tensor([5., 6., 7., 5., 9., 6., 3., 8.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 4)

In [ ]:
test_model = Type3ANFIS(x.shape[1], init_rules=4, x_train=x)

In [ ]:
consequents = test_model.consequents
consequents

Parameter containing:
tensor([[0.7173, 0.7544, 0.6495, 0.6107],
        [0.7049, 0.6934, 0.9890, 0.6170],
        [0.6705, 0.2612, 0.8266, 0.4029],
        [0.0563, 0.4118, 0.3400, 0.7365]], requires_grad=True)

In [ ]:
consequents[0]

tensor([0.7173, 0.7544, 0.6495, 0.6107], grad_fn=<SelectBackward0>)

In [ ]:
test_model.rules

4

In [ ]:
'''
print(test_model.firing_levels(x))
for i in range(test_model.rules):
    print(f"\n - Rule {i}: ")
    xe = torch.cat([x, torch.ones(x.shape[0], 1)], dim=1)
    fls_diag = torch.diag(test_model.firing_levels(x)[:, i])
    print(xe)
    print(fls_diag)
    left = xe.t() @ fls_diag @ xe
    print(f"\nleft: {left}")
    left_i = torch.inverse(left)
    print(left_i)
    #print(left@left_i)
    before_y = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag
    print(before_y)
    print(y)
    new_consequents = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y
    print(new_consequents)
'''

'\nprint(test_model.firing_levels(x))\nfor i in range(test_model.rules):\n    print(f"\n - Rule {i}: ")\n    xe = torch.cat([x, torch.ones(x.shape[0], 1)], dim=1)\n    fls_diag = torch.diag(test_model.firing_levels(x)[:, i])\n    print(xe)\n    print(fls_diag)\n    left = xe.t() @ fls_diag @ xe\n    print(f"\nleft: {left}")\n    left_i = torch.inverse(left)\n    print(left_i)\n    #print(left@left_i)\n    before_y = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag\n    print(before_y)\n    print(y)\n    new_consequents = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y\n    print(new_consequents)\n'

Matrix multiplications are perfect

####Premises

In [ ]:
'''
print(test_model.consequents)
new_consequents = torch.tensor([])
for i in range(test_model.rules):
    print(f"\n - Rule {i}: ")
    xe = torch.cat([x, torch.ones(x.shape[0], 1)], dim=1)
    fls_diag = torch.diag(test_model.firing_levels(x)[:, i])
    new_consequent = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y
    print(new_consequent.unsqueeze(0))
    new_consequents = torch.cat([new_consequents, new_consequent.unsqueeze(0)])

print(f"\nnew consequents: {new_consequents}")
pred = test_model(x)
print(f"\npremises: {test_model.premises}")
print(f"\npremises gradients: {test_model.premises.grad}")

mse = nn.functional.mse_loss(pred, y)
#for i in range(test_model.rules):
print("\nMSE:", mse)
mse.backward()

print(f"\npremises: {test_model.premises}")
print(f"\npremises gradients: {test_model.premises.grad}")
'''

'\nprint(test_model.consequents)\nnew_consequents = torch.tensor([])\nfor i in range(test_model.rules):\n    print(f"\n - Rule {i}: ")\n    xe = torch.cat([x, torch.ones(x.shape[0], 1)], dim=1)\n    fls_diag = torch.diag(test_model.firing_levels(x)[:, i])\n    new_consequent = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y\n    print(new_consequent.unsqueeze(0))\n    new_consequents = torch.cat([new_consequents, new_consequent.unsqueeze(0)])\n\nprint(f"\nnew consequents: {new_consequents}")\npred = test_model(x)\nprint(f"\npremises: {test_model.premises}")\nprint(f"\npremises gradients: {test_model.premises.grad}")\n\nmse = nn.functional.mse_loss(pred, y)\n#for i in range(test_model.rules):\nprint("\nMSE:", mse)\nmse.backward()\n\nprint(f"\npremises: {test_model.premises}")\nprint(f"\npremises gradients: {test_model.premises.grad}")\n'

El gradiente de cada una de las premisas se calcula correctamente con las funciones de pytorch.

In [ ]:
new_consequents = torch.tensor([])
for i in range(test_model.rules):
    xe = torch.cat([x, torch.ones(x.shape[0], 1)], dim=1)
    fls_diag = torch.diag(test_model.firing_levels(x)[:, i])
    new_consequent = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y
    new_consequents = torch.cat([new_consequents, new_consequent.unsqueeze(0)])

pred = test_model(x)
print(f"\npremises gradients: {test_model.premises.grad}")

mse = nn.functional.mse_loss(pred, y)
#for i in range(test_model.rules):
print("\nMSE:", mse)
mse.backward()

print(f"\npremises gradients: {test_model.premises.grad}")




premises gradients: None

MSE: tensor(7.7519, grad_fn=<MseLossBackward0>)

premises gradients: tensor([[[-2.0208e-01, -4.3572e-02],
         [ 8.1452e-06,  3.4936e-05],
         [ 1.6418e-05,  3.2319e-05],
         [-3.3571e-01,  8.4538e-01]],

        [[-1.6933e+00, -8.8654e+00],
         [-1.5678e-05,  5.2372e-07],
         [-6.1668e-05,  2.5055e-04],
         [ 6.1779e-01, -5.0009e+00]],

        [[ 8.3082e-02,  2.6084e-01],
         [ 1.3940e-06, -1.3996e-06],
         [-1.6349e-05,  4.0562e-05],
         [-4.8639e-01,  3.1561e+00]]])


In [ ]:
premises_grads = test_model.premises.grad
torch.sum(torch.pow(premises_grads, 2))

tensor(117.9958)

In [ ]:
y = 0.1 #parameter established by the user
alpha = y/torch.sqrt(torch.sum(torch.pow(premises_grads, 2)))

In [ ]:
alpha

tensor(0.0092)

With the alpha value we can estimate the new premises

In [ ]:
current_premises = test_model.premises.data
current_premises

tensor([[[1.0000, 1.0000],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [7.0000, 1.0000]],

        [[3.0000, 0.5000],
         [4.0000, 0.5000],
         [5.0000, 0.5000],
         [6.0000, 0.5000]],

        [[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333]]])

#####mu parameters (v)

xi - vi

In [ ]:
#mu (v)
vs = current_premises[:,:,0] #this is by x sample dimention
vs

tensor([[1.0000, 3.0000, 5.0000, 7.0000],
        [3.0000, 4.0000, 5.0000, 6.0000],
        [1.0000, 3.6667, 6.3333, 9.0000]])

In [ ]:
''' vi : mu (v) parameter of xi
    x = (x1,...,xN)
[[v1, v2, v3], rule 0
 [v1, v2, v3], rule 1
 [v1, v2, v3], rule 2
 [v1, v2, v3]] rule 3
'''
vs.t()

tensor([[1.0000, 3.0000, 1.0000],
        [3.0000, 4.0000, 3.6667],
        [5.0000, 5.0000, 6.3333],
        [7.0000, 6.0000, 9.0000]])

In [ ]:
x[0]

tensor([5., 5., 1.])

In [ ]:
''' vi : mu (v) parameter of xi
    x = (x1,...,xN)
[[x1 - v1, x2 - v2, x3 - v3], rule 0
 [x1 - v1, x2 - v2, x3 - v3], rule 1
 [x1 - v1, x2 - v2, x3 - v3]] rule 2
'''
x[0] - vs.t()

tensor([[ 4.0000,  2.0000,  0.0000],
        [ 2.0000,  1.0000, -2.6667],
        [ 0.0000,  0.0000, -5.3333],
        [-2.0000, -1.0000, -8.0000]])

4 ̇ alpha ̇  (1/si^2)

In [ ]:
#sigma
sigmas = current_premises[:,:,1]
sigmas

tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [1.3333, 1.3333, 1.3333, 1.3333]])

In [ ]:
''' si : sigma (s) parameter of xi
    x = (x1,...,xN)
[[s1, s2, s3], rule 0
 [s1, s2, s3], rule 1
 [s1, s2, s3], rule 2
 [s1, s2, s3]] rule 3
'''
sigmas.t()

tensor([[1.0000, 0.5000, 1.3333],
        [1.0000, 0.5000, 1.3333],
        [1.0000, 0.5000, 1.3333],
        [1.0000, 0.5000, 1.3333]])

In [ ]:
1/torch.pow(sigmas.t(), 2)

tensor([[1.0000, 4.0000, 0.5625],
        [1.0000, 4.0000, 0.5625],
        [1.0000, 4.0000, 0.5625],
        [1.0000, 4.0000, 0.5625]])

In [ ]:
4*alpha*(1/torch.pow(sigmas.t(), 2))

tensor([[0.0368, 0.1473, 0.0207],
        [0.0368, 0.1473, 0.0207],
        [0.0368, 0.1473, 0.0207],
        [0.0368, 0.1473, 0.0207]])

A = 4 ⋅ alpha ⋅ (1/si^2) ⋅ (xi - vi)^2

In [ ]:
'''
vi : mu (v) parameter of xi
    x = (x1,...,xN)
[[v1, v2, v3], rule 0
 [v1, v2, v3], rule 1
 [v1, v2, v3], rule 2
 [v1, v2, v3]] rule 3

Ai = A calculation for xi with x=(x1,...,xN)
[[A1, A2, A3],  rule 0
 [A1, A2, A3],  rule 1
 [A1, A2, A3],  rule 2
 [A1, A2, A3]]  rule 3
'''
4*alpha*(1/torch.pow(sigmas.t(), 2)) * torch.pow((x[0] - vs.t()), 2)

tensor([[0.5892, 0.5892, 0.0000],
        [0.1473, 0.1473, 0.1473],
        [0.0000, 0.0000, 0.5892],
        [0.1473, 0.1473, 1.3257]])

~wk

In [ ]:
test_model.firing_levels(x)

tensor([[8.8861e+06, 4.0343e+02, 2.9810e+03, 3.5849e+09],
        [1.0597e+09, 1.4528e+03, 3.2416e+02, 1.1772e+07],
        [8.2529e+08, 2.2726e+04, 1.0185e+05, 7.4291e+10],
        [9.0017e+01, 9.0017e+01, 1.4651e+07, 3.8808e+17],
        [1.7619e+13, 3.6315e+04, 1.2182e+01, 6.6514e+02],
        [5.9105e+09, 3.6316e+04, 3.6316e+04, 5.9105e+09]],
       grad_fn=<ProdBackward1>)

In [ ]:
norm_fls = torch.nn.functional.normalize(test_model.firing_levels(x), p=1, dim=-1)
''' ~w = normalization(w)
    wi = rule [i] firing level
[[~w1, ~w2, ~w3, ~w4],  (batch input1)
 [~w1, ~w2, ~w3, ~w4],  (batch input2)
 ...
 [~w1, ~w2, ~w3, ~w4]]  (last batch input)
'''
norm_fls

tensor([[2.4726e-03, 1.1226e-07, 8.2947e-07, 9.9753e-01],
        [9.8901e-01, 1.3559e-06, 3.0254e-07, 1.0987e-02],
        [1.0987e-02, 3.0254e-07, 1.3559e-06, 9.8901e-01],
        [2.3195e-16, 2.3195e-16, 3.7751e-11, 1.0000e+00],
        [1.0000e+00, 2.0612e-09, 6.9144e-13, 3.7751e-11],
        [5.0000e-01, 3.0721e-06, 3.0721e-06, 5.0000e-01]],
       grad_fn=<DivBackward0>)

In [ ]:
norm_fl = torch.nn.functional.normalize(test_model.firing_levels(x[0]), p=1, dim=-1)
''' ~w = normalization(w)
    wi = rule [i] firing level
[~w1, ~w2, ~w3, ~w4]
'''
norm_fl

tensor([2.4726e-03, 1.1226e-07, 8.2947e-07, 9.9753e-01],
       grad_fn=<DivBackward0>)

In [ ]:
norm_fl.unsqueeze(1)

tensor([[2.4726e-03],
        [1.1226e-07],
        [8.2947e-07],
        [9.9753e-01]], grad_fn=<UnsqueezeBackward0>)

B = A * ~wk

In [ ]:
4*alpha*(1/torch.pow(sigmas.t(), 2)) * torch.pow((x[0] - vs.t()), 2)

tensor([[0.5892, 0.5892, 0.0000],
        [0.1473, 0.1473, 0.1473],
        [0.0000, 0.0000, 0.5892],
        [0.1473, 0.1473, 1.3257]])

In [ ]:
norm_fl.unsqueeze(1)

tensor([[2.4726e-03],
        [1.1226e-07],
        [8.2947e-07],
        [9.9753e-01]], grad_fn=<UnsqueezeBackward0>)

In [ ]:
'''
Bi = B calculation for xi with x=(x1,...,xN)
[[B1, B2, B3],  rule 0
 [B1, B2, B3],  rule 1
 [B1, B2, B3],  rule 2
 [B1, B2, B3]]  rule 3
'''
B = 4*alpha*(1/torch.pow(sigmas.t(), 2)) * torch.pow((x[0] - vs.t()), 2) * norm_fl.unsqueeze(1)
B

tensor([[1.4568e-03, 1.4568e-03, 0.0000e+00],
        [1.6535e-08, 1.6535e-08, 1.6535e-08],
        [0.0000e+00, 0.0000e+00, 4.8871e-07],
        [1.4693e-01, 1.4693e-01, 1.3224e+00]], grad_fn=<MulBackward0>)

zk = (fk - g)(y - g)

In [ ]:
fks = test_model.outputs_by_rule(x)
fks

tensor([[2.1311e-02, 9.6517e-07, 4.8839e-06, 3.4085e+00],
        [9.8761e+00, 1.3955e-05, 2.2343e-06, 4.1896e-02],
        [1.1529e-01, 3.5058e-06, 1.2456e-05, 3.6850e+00],
        [1.5863e-15, 1.9362e-15, 2.2613e-10, 3.7282e+00],
        [1.3098e+01, 3.2577e-08, 8.1799e-12, 2.2956e-10],
        [4.3280e+00, 2.6378e-05, 1.6831e-05, 1.8862e+00]],
       grad_fn=<MulBackward0>)

In [ ]:
fk = test_model.outputs_by_rule(x[0])
fk

tensor([2.1311e-02, 9.6517e-07, 4.8839e-06, 3.4085e+00],
       grad_fn=<MulBackward0>)

In [ ]:
'''fk for x[0]
[[f1],
 [f2],
 [f3],
 [f4]]
'''
fk.unsqueeze(1)

tensor([[2.1311e-02],
        [9.6517e-07],
        [4.8839e-06],
        [3.4085e+00]], grad_fn=<UnsqueezeBackward0>)

In [ ]:
g = pred
g[0]

tensor(3.4298, grad_fn=<SelectBackward0>)

In [ ]:
Y = torch.tensor([5., 6., 7., 5., 9., 6.])
Y[0]

tensor(5.)

In [ ]:
Y[0] - g[0]

tensor(1.5702, grad_fn=<SubBackward0>)

In [ ]:
fk.unsqueeze(1) - g[0]

tensor([[-3.4085],
        [-3.4298],
        [-3.4298],
        [-0.0213]], grad_fn=<SubBackward0>)

In [ ]:
z = (fk.unsqueeze(1) - g[0])*(Y[0] - g[0])
z

tensor([[-5.3520],
        [-5.3855],
        [-5.3855],
        [-0.0335]], grad_fn=<MulBackward0>)

new premises for x[0]

In [ ]:
B

tensor([[1.4568e-03, 1.4568e-03, 0.0000e+00],
        [1.6535e-08, 1.6535e-08, 1.6535e-08],
        [0.0000e+00, 0.0000e+00, 4.8871e-07],
        [1.4693e-01, 1.4693e-01, 1.3224e+00]], grad_fn=<MulBackward0>)

In [ ]:
z

tensor([[-5.3520],
        [-5.3855],
        [-5.3855],
        [-0.0335]], grad_fn=<MulBackward0>)

In [ ]:
''' vi : mu (v) parameter of xi
    x = (x1,...,xN)
[[v1, v2, v3], rule 0
 [v1, v2, v3], rule 1
 [v1, v2, v3], rule 2
 [v1, v2, v3]] rule 3
'''
new_v0 = B*z
new_v0

tensor([[-7.7969e-03, -7.7969e-03, -0.0000e+00],
        [-8.9048e-08, -8.9048e-08, -8.9048e-08],
        [-0.0000e+00, -0.0000e+00, -2.6319e-06],
        [-4.9181e-03, -4.9181e-03, -4.4263e-02]], grad_fn=<MulBackward0>)

so the premises structure changes

In [ ]:
new_v0 = new_v0.t()
new_v0

tensor([[-7.7969e-03, -8.9048e-08, -0.0000e+00, -4.9181e-03],
        [-7.7969e-03, -8.9048e-08, -0.0000e+00, -4.9181e-03],
        [-0.0000e+00, -8.9048e-08, -2.6319e-06, -4.4263e-02]],
       grad_fn=<TBackward0>)

In [ ]:
new_prem = torch.zeros_like(current_premises)
new_prem[:,:,0] = new_v0
new_prem

tensor([[[-7.7969e-03,  0.0000e+00],
         [-8.9048e-08,  0.0000e+00],
         [-0.0000e+00,  0.0000e+00],
         [-4.9181e-03,  0.0000e+00]],

        [[-7.7969e-03,  0.0000e+00],
         [-8.9048e-08,  0.0000e+00],
         [-0.0000e+00,  0.0000e+00],
         [-4.9181e-03,  0.0000e+00]],

        [[-0.0000e+00,  0.0000e+00],
         [-8.9048e-08,  0.0000e+00],
         [-2.6319e-06,  0.0000e+00],
         [-4.4263e-02,  0.0000e+00]]], grad_fn=<CopySlices>)

What will happen when the entire training data set is considered?

#####sigma parameters

(xi - vi)

In [ ]:
x[0] - vs.t()

tensor([[ 4.0000,  2.0000,  0.0000],
        [ 2.0000,  1.0000, -2.6667],
        [ 0.0000,  0.0000, -5.3333],
        [-2.0000, -1.0000, -8.0000]])

4⋅alpha⋅(1/sigma^3)

In [ ]:
4*alpha*(1/torch.pow(sigmas.t(), 3))

tensor([[0.0368, 0.2946, 0.0155],
        [0.0368, 0.2946, 0.0155],
        [0.0368, 0.2946, 0.0155],
        [0.0368, 0.2946, 0.0155]])

4⋅alpha⋅(1/sigma^3)⋅(xi - vi)

In [ ]:
4*alpha*(1/torch.pow(sigmas.t(), 3))*(x[0] - vs.t())

tensor([[ 0.1473,  0.5892,  0.0000],
        [ 0.0736,  0.2946, -0.0414],
        [ 0.0000,  0.0000, -0.0829],
        [-0.0736, -0.2946, -0.1243]])

~wk

In [ ]:
'''~wk for x[0]
[[~w1],
 [~w2],
 [~w3],
 [~w4]]
'''
norm_fl.unsqueeze(1)

tensor([[2.4726e-03],
        [1.1226e-07],
        [8.2947e-07],
        [9.9753e-01]], grad_fn=<UnsqueezeBackward0>)

zk = (fk - g)(y - g)

In [ ]:
'''fk for x[0]
[[f1],
 [f2],
 [f3],
 [f4]]
'''
fk.unsqueeze(1)

tensor([[2.1311e-02],
        [9.6517e-07],
        [4.8839e-06],
        [3.4085e+00]], grad_fn=<UnsqueezeBackward0>)

In [ ]:
g[0]

tensor(3.4298, grad_fn=<SelectBackward0>)

In [ ]:
Y[0]

tensor(5.)

In [ ]:
z = (fk.unsqueeze(1) - g[0])*(Y[0] - g[0])
z

tensor([[-5.3520],
        [-5.3855],
        [-5.3855],
        [-0.0335]], grad_fn=<MulBackward0>)

4⋅alpha⋅(1/sigma^3)⋅(xi - vi)⋅~w⋅z

In [ ]:
new_s0 = 4*alpha*(1/torch.pow(sigmas.t(), 3))*(x[0] - vs.t())*norm_fl.unsqueeze(1)*z
new_s0

tensor([[-1.9492e-03, -7.7969e-03, -0.0000e+00],
        [-4.4524e-08, -1.7810e-07,  2.5045e-08],
        [-0.0000e+00, -0.0000e+00,  3.7011e-07],
        [ 2.4591e-03,  9.8362e-03,  4.1497e-03]], grad_fn=<MulBackward0>)

so the premises structure changes

In [ ]:
new_s0 = new_s0.t()
new_s0

tensor([[-1.9492e-03, -4.4524e-08, -0.0000e+00,  2.4591e-03],
        [-7.7969e-03, -1.7810e-07, -0.0000e+00,  9.8362e-03],
        [-0.0000e+00,  2.5045e-08,  3.7011e-07,  4.1497e-03]],
       grad_fn=<TBackward0>)

In [ ]:
new_prem[:,:,1] = new_s0
new_prem

tensor([[[-7.7969e-03, -1.9492e-03],
         [-8.9048e-08, -4.4524e-08],
         [-0.0000e+00, -0.0000e+00],
         [-4.9181e-03,  2.4591e-03]],

        [[-7.7969e-03, -7.7969e-03],
         [-8.9048e-08, -1.7810e-07],
         [-0.0000e+00, -0.0000e+00],
         [-4.9181e-03,  9.8362e-03]],

        [[-0.0000e+00, -0.0000e+00],
         [-8.9048e-08,  2.5045e-08],
         [-2.6319e-06,  3.7011e-07],
         [-4.4263e-02,  4.1497e-03]]], grad_fn=<CopySlices>)

#####Premises update

In [ ]:
current_premises

tensor([[[1.0000, 1.0000],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [7.0000, 1.0000]],

        [[3.0000, 0.5000],
         [4.0000, 0.5000],
         [5.0000, 0.5000],
         [6.0000, 0.5000]],

        [[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333]]])

In [ ]:
new_prem

tensor([[[-7.7969e-03, -1.9492e-03],
         [-8.9048e-08, -4.4524e-08],
         [-0.0000e+00, -0.0000e+00],
         [-4.9181e-03,  2.4591e-03]],

        [[-7.7969e-03, -7.7969e-03],
         [-8.9048e-08, -1.7810e-07],
         [-0.0000e+00, -0.0000e+00],
         [-4.9181e-03,  9.8362e-03]],

        [[-0.0000e+00, -0.0000e+00],
         [-8.9048e-08,  2.5045e-08],
         [-2.6319e-06,  3.7011e-07],
         [-4.4263e-02,  4.1497e-03]]], grad_fn=<CopySlices>)

In [ ]:
current_premises

tensor([[[1.0000, 1.0000],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [7.0000, 1.0000]],

        [[3.0000, 0.5000],
         [4.0000, 0.5000],
         [5.0000, 0.5000],
         [6.0000, 0.5000]],

        [[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333]]])

In [ ]:
new_premises0 = current_premises + new_prem
new_premises0

tensor([[[0.9922, 0.9981],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [6.9951, 1.0025]],

        [[2.9922, 0.4922],
         [4.0000, 0.5000],
         [5.0000, 0.5000],
         [5.9951, 0.5098]],

        [[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [8.9557, 1.3375]]], grad_fn=<AddBackward0>)

#####Cycle of premises

In [ ]:
pred

tensor([ 3.4298,  9.9180,  3.8003,  3.7282, 13.0980,  6.2142],
       grad_fn=<SumBackward1>)

In [ ]:
Y

tensor([5., 6., 7., 5., 9., 6.])

In [ ]:
x

tensor([[5., 5., 1.],
        [6., 5., 2.],
        [7., 3., 4.],
        [1., 3., 5.],
        [4., 5., 9.],
        [4., 6., 1.]])

In [ ]:
sigmas

tensor([[1.0000, 1.0000, 1.0000, 1.0000],
        [0.5000, 0.5000, 0.5000, 0.5000],
        [1.3333, 1.3333, 1.3333, 1.3333]])

In [ ]:
vs

tensor([[1.0000, 3.0000, 5.0000, 7.0000],
        [3.0000, 4.0000, 5.0000, 6.0000],
        [1.0000, 3.6667, 6.3333, 9.0000]])

In [ ]:
x_train[0]

tensor([5.3262, 9.9451])

In [ ]:
current_premises = test_model.premises.data
sigmas = current_premises[:,:,1].t()
vs = current_premises[:,:,0].t()

total_to_add = torch.zeros_like(current_premises)
for sample, target, g in zip(x, Y, pred):
    A = 4*alpha*(1/torch.pow(sigmas, 2)) * (sample - vs) #A = 4 ⋅ alpha ⋅ (1/si^2) ⋅ (xi - vi)^2
    w = torch.nn.functional.normalize(test_model.firing_levels(sample), p=1, dim=-1).unsqueeze(1)
    f = test_model.outputs_by_rule(sample).unsqueeze(1)
    z = (f - g)*(target - g)
    vs_to_add = A*w*z
    total_to_add[:,:,0] += vs_to_add.t()
    #print(f"\nvs: {vs_to_add.t()}")

    B = 4*alpha*(1/torch.pow(sigmas, 3)) * torch.pow((sample - vs), 2)
    sigmas_to_add = B*w*z
    total_to_add[:,:,1] += sigmas_to_add.t()
    #print(f"\nsigmas: {sigmas_to_add.t()}")

    #print(f"\ntotal to add: {total_to_add}")

new_premises = current_premises + total_to_add
print(current_premises)
print(new_premises)

tensor([[[1.0000, 1.0000],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [7.0000, 1.0000]],

        [[3.0000, 0.5000],
         [4.0000, 0.5000],
         [5.0000, 0.5000],
         [6.0000, 0.5000]],

        [[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333]]])
tensor([[[ 1.0217,  1.0369],
         [ 3.0000,  1.0000],
         [ 5.0000,  1.0000],
         [ 6.9356,  1.1644]],

        [[ 3.1332,  1.2115],
         [ 4.0000,  0.5000],
         [ 5.0000,  0.5000],
         [ 6.1035, -0.3520]],

        [[ 0.9953,  1.3177],
         [ 3.6667,  1.3333],
         [ 6.3333,  1.3333],
         [ 8.9049,  1.9430]]], grad_fn=<AddBackward0>)


####The Proposal

In [ ]:
def OLS(ANFISmodel, x_train, y_train, y):
    #iterates by rule
    for i in range(ANFISmodel.rules):
        #needed tensors for consequents calculation
        xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1)
        fls_diag = torch.diag(ANFISmodel.firing_levels(x_train)[:, i])

        #new consequents
        new_consequents = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y_train

    #models prediction
    pred = ANFISmodel(x_train)

    #mse and premises gradients calculation with backward pytorch function
    mse = nn.functional.mse_loss(pred, y_train)
    mse.backward()

    #alpha calculation (with premises gradients)
    alpha = y/torch.sqrt(torch.sum(torch.pow(ANFISmodel.premises.grad, 2)))

    #premises extracted
    current_premises = ANFISmodel.premises.data
    sigmas = current_premises[:,:,1].t()
    vs = current_premises[:,:,0].t()

    #tensor to acumulate the premises updates
    to_add = torch.zeros_like(current_premises)

    #loop over the train set and models prediction
    for sample, target, g in zip(x_train, y_train, pred):
        #necessary calculations for updating vs and sigmas parameters
        A = 4 * alpha * (1/torch.pow(sigmas, 2)) * (sample - vs) #A = 4 ⋅ alpha ⋅ (1/si^2) ⋅ (xi - vi)
        B = 4 * alpha * (1/torch.pow(sigmas, 3)) * torch.pow((sample - vs), 2) #B = 4 ⋅ alpha ⋅ (1/si^3) ⋅ (xi - vi)^2
        w = torch.nn.functional.normalize(ANFISmodel.firing_levels(sample), p=1, dim=-1).unsqueeze(1)
        f = test_model.outputs_by_rule(sample).unsqueeze(1)
        z = (f - g)*(target - g)

        #vs and sigmas parameters updates
        vs_to_add = A*w*z
        sigmas_to_add = B*w*z

        #values updated
        to_add[:,:,0] += vs_to_add.t()
        to_add[:,:,1] += sigmas_to_add.t()

    #new premises
    new_premises = current_premises + to_add

    ANFISmodel.set_premises(Parameter(new_premises, requires_grad=True))
    ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

####Testing

###Proposal 2 (1 with a litlle change)
Extracts the dataset from a dataloader

In [ ]:
x_train = torch.tensor([[5., 5., 1.],
                  [6., 5., 2.],
                  [7., 3., 4.],
                  [1., 3., 5.],
                  [4., 5., 9.],
                  [4., 6., 1.]])
y_train = torch.tensor([5., 6., 7., 5., 9., 6.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 4)

In [ ]:
data_set = test_loader.dataset

In [ ]:
x_train = data_set.tensors[0]
x_train

tensor([[5., 5., 1.],
        [6., 5., 2.],
        [7., 3., 4.],
        [1., 3., 5.],
        [4., 5., 9.],
        [4., 6., 1.]])

In [ ]:
y_train = data_set.tensors[1]
y_train

tensor([5., 6., 7., 5., 9., 6.])

In [ ]:
test_model = Type3ANFIS(x_train.shape[1], init_rules=4, x_train=x_train)

####The Proposal

In [ ]:
def OLS(ANFISmodel, loader, y):
    x_train = test_loader.dataset.tensors[0]
    y_train = test_loader.dataset.tensors[1]
    #iterates by rule
    for i in range(ANFISmodel.rules):
        #needed tensors for consequents calculation
        xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1)
        fls_diag = torch.diag(ANFISmodel.firing_levels(x_train)[:, i])

        #new consequents
        new_consequents = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y_train

    #models prediction
    pred = ANFISmodel(x_train)

    #mse and premises gradients calculation with backward pytorch function
    mse = nn.functional.mse_loss(pred, y_train)
    mse.backward()

    #alpha calculation (with premises gradients)
    alpha = y / torch.sqrt(torch.sum(torch.pow(ANFISmodel.premises.grad, 2)))

    #premises extracted
    current_premises = ANFISmodel.premises.data
    sigmas = current_premises[:,:,1].t()
    vs = current_premises[:,:,0].t()

    #tensor to acumulate the premises updates
    to_add = torch.zeros_like(current_premises)

    #loop over the train set and models prediction
    for sample, target, g in zip(x_train, y_train, pred):
        #necessary calculations for updating vs and sigmas parameters
        A = 4 * alpha * (1/torch.pow(sigmas, 2)) * (sample - vs)
        B = 4 * alpha * (1/torch.pow(sigmas, 3)) * torch.pow((sample - vs), 2)
        w = torch.nn.functional.normalize(ANFISmodel.firing_levels(sample), p=1, dim=-1).unsqueeze(1)
        f = test_model.outputs_by_rule(sample).unsqueeze(1)
        z = (f - g)*(target - g)

        #vs and sigmas parameters updates
        vs_to_add = A*w*z
        sigmas_to_add = B*w*z

        #values updated
        to_add[:,:,0] += vs_to_add.t()
        to_add[:,:,1] += sigmas_to_add.t()

    #new premises
    new_premises = current_premises + to_add

    ANFISmodel.set_premises(Parameter(new_premises, requires_grad=True))
    ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

###Proposal 3
Premises estimation with the entire data set

####Consequents

In [ ]:
x = torch.tensor([[5., 5., 1.],
                  [6., 5., 2.],
                  [7., 3., 4.],
                  [1., 3., 5.],
                  [4., 5., 9.],
                  [4., 6., 1.]])
y = torch.tensor([5., 6., 7., 5., 9., 6.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 4)

In [ ]:
test_model = Type3ANFIS(x.shape[1], init_rules=4, x_train=x)

In [ ]:
x_train = test_loader.dataset.tensors[0]
y_train = test_loader.dataset.tensors[1]

In [ ]:
#needed tensors for consequents calculation
new_consequents = torch.zeros_like(test_model.consequents.data)
xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1)

for i in range(test_model.rules):
    #firing_levels diagonal matrix
    fls_diag = torch.diag(test_model.firing_levels(x_train)[:, i])

    #new consequents
    new_consequents[i] = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y_train

####Premises

In [ ]:
pred = test_model(x_train)

In [ ]:
mse = nn.functional.mse_loss(pred, y_train)
mse.backward()

In [ ]:
alpha = 0.1 / torch.sqrt(torch.sum(torch.pow(test_model.premises.grad, 2)))

In [ ]:
current_premises = test_model.premises.data
sigmas = current_premises[:,:,1].t()
vs = current_premises[:,:,0].t()

In [ ]:
current_premises

tensor([[[1.0000, 1.0000],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [7.0000, 1.0000]],

        [[3.0000, 0.5000],
         [4.0000, 0.5000],
         [5.0000, 0.5000],
         [6.0000, 0.5000]],

        [[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333]]])

In [ ]:
vs = current_premises[:, :, 0].t()
vs

tensor([[1.0000, 3.0000, 1.0000],
        [3.0000, 4.0000, 3.6667],
        [5.0000, 5.0000, 6.3333],
        [7.0000, 6.0000, 9.0000]])

In [ ]:
sigmas = current_premises[:, :, 1].t()
sigmas

tensor([[1.0000, 0.5000, 1.3333],
        [1.0000, 0.5000, 1.3333],
        [1.0000, 0.5000, 1.3333],
        [1.0000, 0.5000, 1.3333]])

In [ ]:
#vs
print(f"\n\nRule {0}:")
A = 4*alpha*(1/torch.pow(sigmas[0], 2))*(x_train - vs[0])
wk = torch.nn.functional.normalize(test_model.firing_levels(x_train), p=1, dim=-1)[:,0].unsqueeze(0).t()
fk = test_model.outputs_by_rule(x_train)[:,0]
zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()
print(f"\nRule {0} to_add vs :\n{torch.sum(A*wk*zk, dim=0)}")

#sigmas
B = 4*alpha*(1/torch.pow(sigmas[0], 3))*torch.pow((x_train - vs[0]), 2)
print(f"\nRule {0} to_add sigmas :\n{torch.sum(B*wk*zk, dim=0)}")



Rule 0:

Rule 0 to_add vs :
tensor([0.4804, 1.4902, 0.0247], grad_fn=<SumBackward1>)

Rule 0 to_add sigmas :
tensor([1.7724, 8.6690, 0.0438], grad_fn=<SumBackward1>)


In [ ]:
new_vs = torch.zeros_like(vs)
new_vs

tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])

In [ ]:
vs = current_premises[:, :, 0].t()
sigmas = current_premises[:, :, 1].t()
new_vs = torch.zeros_like(vs)
new_sigmas = torch.zeros_like(sigmas)
current_premises2 = current_premises

In [ ]:
for i in range(test_model.rules):
    #vs
    #print(f"\n\nRule {i}:")
    A = 4*alpha*(1/torch.pow(sigmas[i], 2))*(x_train - vs[i])
    wk = torch.nn.functional.normalize(test_model.firing_levels(x_train), p=1, dim=-1)[:,i].unsqueeze(0).t()
    fk = test_model.outputs_by_rule(x_train)[:,i]
    zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()
    #print(f"\nRule {i} to_add vs :\n{torch.sum(A*wk*zk, dim=0)}")

    #sigmas
    B = 4*alpha*(1/torch.pow(sigmas[i], 3))*torch.pow((x_train - vs[i]), 2)
    #print(f"\nRule {i} to_add sigmas :\n{torch.sum(B*wk*zk, dim=0)}")

    new_vs[i] = torch.sum(A*wk*zk, dim=0)
    new_sigmas[i] = torch.sum(B*wk*zk, dim=0)

#print(f"new_vs.t() :\n{new_vs.t()}")
#print(f"new_sigmas.t() :\n{new_sigmas.t()}")

print(f"\ncurrent premises :\n{current_premises}")

current_premises[:, :, 0] += new_vs.t()
current_premises[:, :, 1] += new_sigmas.t()

print(f"\ncurrent premises :\n{current_premises}")


current premises :
tensor([[[1.0000, 1.0000],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [7.0000, 1.0000]],

        [[3.0000, 0.5000],
         [4.0000, 0.5000],
         [5.0000, 0.5000],
         [6.0000, 0.5000]],

        [[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333]]])

current premises :
tensor([[[1.4804, 2.7724],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [6.6867, 1.9073]],

        [[4.4902, 9.1690],
         [4.0000, 0.5001],
         [5.0000, 0.5001],
         [5.8105, 1.3757]],

        [[1.0247, 1.3772],
         [3.6667, 1.3333],
         [6.3333, 1.3334],
         [8.4639, 4.4449]]], grad_fn=<CopySlices>)


In [ ]:
for i in range(test_model.rules):
    #vs
    #print(f"\n\nRule {i}:")
    A = 4*alpha*(1/torch.pow(sigmas[i], 2))*(x_train - vs[i])
    wk = torch.nn.functional.normalize(test_model.firing_levels(x_train), p=1, dim=-1)[:,i].unsqueeze(0).t()
    fk = test_model.outputs_by_rule(x_train)[:,i]
    zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()
    #print(f"\nRule {i} to_add vs :\n{torch.sum(A*wk*zk, dim=0)}")

    #sigmas
    B = 4*alpha*(1/torch.pow(sigmas[i], 3))*torch.pow((x_train - vs[i]), 2)
    #print(f"\nRule {i} to_add sigmas :\n{torch.sum(B*wk*zk, dim=0)}")

    new_vs[i] = torch.sum(A*wk*zk, dim=0)
    new_sigmas[i] = torch.sum(B*wk*zk, dim=0)

#print(f"new_vs.t() :\n{new_vs.t()}")
#print(f"new_sigmas.t() :\n{new_sigmas.t()}")

print(f"\ncurrent premises :\n{current_premises}")

current_premises[:, :, 0] += new_vs.t()
current_premises[:, :, 1] += new_sigmas.t()

print(f"\ncurrent premises :\n{current_premises}")


current premises :
tensor([[[1.4804, 2.7724],
         [3.0000, 1.0000],
         [5.0000, 1.0000],
         [6.6867, 1.9073]],

        [[4.4902, 9.1690],
         [4.0000, 0.5001],
         [5.0000, 0.5001],
         [5.8105, 1.3757]],

        [[1.0247, 1.3772],
         [3.6667, 1.3333],
         [6.3333, 1.3334],
         [8.4639, 4.4449]]], grad_fn=<CopySlices>)

current premises :
tensor([[[ 1.4829,  2.7766],
         [ 4.2066,  4.6840],
         [ 5.9147,  2.5055],
         [ 6.6849,  1.9086]],

        [[ 4.4902,  9.1690],
         [ 5.2010,  6.5627],
         [ 2.6321, 13.0235],
         [ 5.8081,  1.3773]],

        [[ 1.0227,  1.3541],
         [ 3.2416,  2.1470],
         [ 3.6014, 10.8272],
         [ 8.4620,  4.4479]]], grad_fn=<CopySlices>)


####The Proposal

In [ ]:
def OLS(ANFISmodel, loader, y):
    #---CONSEQUENTS ESTIMATION---

    #needed tensors for consequents calculation
    x_train = test_loader.dataset.tensors[0]
    new_consequents = torch.zeros_like(ANFISmodel.consequents.data)
    xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1) #extended x_train
    fls = ANFISmodel.firing_levels(x_train) #firing_levels

    #iterates by rule
    for i in range(ANFISmodel.rules):
        #firing_levels diagonal matrix
        fls_diag = torch.diag(fls[:, i])

        #new consequents
        new_consequents[i] = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y_train


    #---PREMISES ESTIMATION---

    #needed tensors for consequents calculation
    y_train = test_loader.dataset.tensors[1] #targets
    pred = ANFISmodel(x_train) #models prediction

    #mse and premises gradients calculation with pytorchs backward function
    mse = nn.functional.mse_loss(pred, y_train)
    mse.backward()

    #alpha calculation (with premises gradients obtained before with mse.backward())
    alpha = y / torch.sqrt(torch.sum(torch.pow(ANFISmodel.premises.grad, 2)))

    #premises extracted
    current_premises = ANFISmodel.premises.data
    vs = current_premises[:,:,0].t()
    sigmas = current_premises[:,:,1].t()
    new_vs = torch.zeros_like(vs)
    new_sigmas = torch.zeros_like(sigmas)

    #iterates by rule
    for i in range(ANFISmodel.rules):
        #mu
        A = 4*alpha*(1/torch.pow(sigmas[i], 2))*(x_train - vs[i])
        wk = torch.nn.functional.normalize(fls, p=1, dim=-1)[:,i].unsqueeze(0).t()
        fk = ANFISmodel.outputs_by_rule(x_train)[:,i]
        zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()

        new_vs[i] = torch.sum(A*wk*zk, dim=0)

        #sigma
        B = 4*alpha*(1/torch.pow(sigmas[i], 3))*torch.pow((x_train - vs[i]), 2)

        new_sigmas[i] = torch.sum(B*wk*zk, dim=0)

    #Premises update
    current_premises[:, :, 0] += new_vs.t()
    current_premises[:, :, 1] += new_sigmas.t()

    #Parameters set on ANFISmodel
    ANFISmodel.set_premises(Parameter(current_premises, requires_grad=True))
    ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

###Proposal 4
No iteration, only tensor operations

In [ ]:
x = torch.tensor([[5., 5., 1.],
                  [6., 5., 2.],
                  [7., 3., 4.],
                  [1., 3., 5.],
                  [4., 5., 9.],
                  [4., 6., 1.]])
y = torch.tensor([5., 6., 7., 5., 9., 6.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 4)

In [ ]:
test_model = Type3ANFIS(x.shape[1], init_rules=4, x_train=x)

In [ ]:
x_train = test_loader.dataset.tensors[0]
y_train = test_loader.dataset.tensors[1]

####Consequents

In [ ]:
new_consequents = torch.zeros_like(test_model.consequents.data)
xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1)

In [ ]:
diagonals = torch.eye(test_model.rules).unsqueeze(0).expand(x_train.shape[0], -1, -1)

In [ ]:
fl_diags = test_model.firing_levels(x_train).unsqueeze(1) * diagonals

In [ ]:
xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1)
xe = xe.unsqueeze(1)
xe_t = xe.unsqueeze(2)

In [ ]:
xe_t @ fl_diags

tensor([[[[4.4431e+07, 2.0171e+03, 2.9810e+03, 3.5849e+09]],

         [[5.2985e+09, 7.2640e+03, 3.2416e+02, 1.1772e+07]],

         [[4.1265e+09, 1.1363e+05, 1.0185e+05, 7.4291e+10]],

         [[4.5009e+02, 4.5009e+02, 1.4651e+07, 3.8808e+17]],

         [[8.8095e+13, 1.8158e+05, 1.2182e+01, 6.6514e+02]],

         [[2.9553e+10, 1.8158e+05, 3.6316e+04, 5.9105e+09]]],


        [[[5.3317e+07, 2.0171e+03, 5.9619e+03, 3.5849e+09]],

         [[6.3582e+09, 7.2640e+03, 6.4833e+02, 1.1772e+07]],

         [[4.9518e+09, 1.1363e+05, 2.0370e+05, 7.4291e+10]],

         [[5.4010e+02, 4.5009e+02, 2.9301e+07, 3.8808e+17]],

         [[1.0571e+14, 1.8158e+05, 2.4365e+01, 6.6514e+02]],

         [[3.5463e+10, 1.8158e+05, 7.2631e+04, 5.9105e+09]]],


        [[[6.2203e+07, 1.2103e+03, 1.1924e+04, 3.5849e+09]],

         [[7.4179e+09, 4.3584e+03, 1.2967e+03, 1.1772e+07]],

         [[5.7771e+09, 6.8177e+04, 4.0740e+05, 7.4291e+10]],

         [[6.3012e+02, 2.7005e+02, 5.8603e+07, 3.8808e+17]],

    

###IMPORTANT: About matrix inverse
If there is a case when the matrix inverse calculation
```
torch.inverse(xe.t() @ fls_diag @ xe)
```
for the consequents estimation is not posible, may it be because the rules are too relative?, since the matrix xe.t()@fls_diag@xe becomes singular (
linear dependence).  
**Should there be some mechanism to resolve this somehow?**


# SONFIS operators
At the moment, these operators are only implemented for the gaussian2 function (used as an MF with 2 premises parameters)

##Best Matching Criteria

###Proposal 1
It may be better to had a special function to calculate the best matching criteria given a data set.

In [ ]:
#def BestMatchingCriteria(ANFISmodel, loader):

##Grow Net

###Proposal 1
Uses a DataLoader of Pytorch to operate batch by batch and a dictionary to cluster the bad samples by rule. The best matching criteria its calculated in the GrowNet function.

In [ ]:
x = torch.tensor([[1, 2],[4, 5],[7, 8],[10, 11]])
x

tensor([[ 1,  2],
        [ 4,  5],
        [ 7,  8],
        [10, 11]])

In [ ]:
w = torch.tensor([[6, 5, 8],[2, 4, 6],[7, 7, 5],[9, 1, 2]])
''' wi = feature [i] firing level
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
w

tensor([[6, 5, 8],
        [2, 4, 6],
        [7, 7, 5],
        [9, 1, 2]])

In [ ]:
max_w = torch.max(w, dim=1)
max_w

torch.return_types.max(
values=tensor([8, 6, 7, 9]),
indices=tensor([2, 2, 0, 0]))

In [ ]:
mask = (max_w.values <= 7)

In [ ]:
cond = max_w.values[mask]
cond

tensor([6, 7])

In [ ]:
bad_samples = x[mask]
bad_samples

tensor([[4, 5],
        [7, 8]])

In [ ]:
bad_samples_rules = max_w.indices[mask]
bad_samples_rules

tensor([2, 0])

In [ ]:
dicc = {}
dicc[bad_samples_rules] = x[mask]

In [ ]:
unique_rules, counts = torch.unique(bad_samples_rules, return_counts=True)

In [ ]:
unique_rules

tensor([0, 2])

In [ ]:
counts

tensor([1, 1])

In [ ]:
a = torch.tensor([])
a

tensor([])

In [ ]:
a = torch.cat((a, bad_samples), dim=0)

In [ ]:
a

tensor([[4., 5.],
        [7., 8.]])

In [ ]:
b = torch.tensor([[4., 5.],
        [7., 8.]])

In [ ]:
a = torch.cat((a, b), dim=0)
a

tensor([[4., 5.],
        [7., 8.],
        [4., 5.],
        [7., 8.]])

In [ ]:
dic = {0: torch.tensor([]), 1: torch.tensor([]), 2: torch.tensor([])}
bs = torch.tensor([[1., 2.],
                   [3., 4.],
                   [5., 6.],
                   [7., 8.],
                   [9., 10.],
                   [11., 12.]])
rules = torch.tensor([2, 0, 2, 2, 1, 0])

for row_index, rule in enumerate(rules):
    row = bs[row_index, :].unsqueeze(0)
    #print(row)

    if rule.item() in dic:
        dic[rule.item()] = torch.cat((dic[rule.item()], row), dim=0)
    else:
        dic[rule.item()] = row

print(dic)

{0: tensor([[ 3.,  4.],
        [11., 12.]]), 1: tensor([[ 9., 10.]]), 2: tensor([[1., 2.],
        [5., 6.],
        [7., 8.]])}


In [ ]:
dic = {rule: samples for rule, samples in dic.items() if samples.size(0) >= 2}
dic

{0: tensor([[ 3.,  4.],
         [11., 12.]]),
 2: tensor([[1., 2.],
         [5., 6.],
         [7., 8.]])}

In [ ]:
'''
x1 premises
[[[mu, sigma, f],   (feature 1)
  [mu, sigma, f],   (feature 2)
  [mu, sigma, f]],  (feature 3)

x2 premises
 [[mu, sigma, f],   (feature 1)
  [mu, sigma, f],   (feature 2)
  [mu, sigma, f]]]  (feature 3)
'''
premises = torch.tensor([[[1, 2],
         [2, 3],
         [3, 4]],

        [[2, 4],
         [4, 6],
         [6, 8]]])

In [ ]:
means = dic[0].mean(dim=0)
means

tensor([7., 8.])

In [ ]:
stds = dic[0].std(dim=0)
stds

tensor([5.6569, 5.6569])

In [ ]:
new_rule = torch.cat([means.unsqueeze(0), stds.unsqueeze(0)])
new_rule

tensor([[7.0000, 8.0000],
        [5.6569, 5.6569]])

In [ ]:
premises = torch.tensor([[[1, 2],
                          [4, 5],
                          [7, 8]],

                         [[2, 4],
                          [1, 3],
                          [6, 8]]])

means = torch.tensor([10, 20])
stds = torch.tensor([100, 200])

means_expanded = means.unsqueeze(1)
stds_expanded = stds.unsqueeze(1)

result = torch.cat((means_expanded, stds_expanded), dim=1)
print(result.unsqueeze(1))

result_tensor = torch.cat((premises, result.unsqueeze(1)), dim=1)
print(result_tensor)

tensor([[[ 10, 100]],

        [[ 20, 200]]])
tensor([[[  1,   2],
         [  4,   5],
         [  7,   8],
         [ 10, 100]],

        [[  2,   4],
         [  1,   3],
         [  6,   8],
         [ 20, 200]]])


In [ ]:
premises = torch.tensor([[[1, 2, 3],
                          [4, 5, 6],
                          [7, 8, 9]],

                         [[2, 4, 6],
                          [1, 3, 5],
                          [6, 8, 10]]])
'''
        x1 premises
        [[[mu, sigma, f],   (feature 1)
          [mu, sigma, f],   (feature 2)
          [mu, sigma, f]],  (feature 3)

        x2 premises
         [[mu, sigma, f],   (feature 1)
          [mu, sigma, f],   (feature 2)
          [mu, sigma, f]]]  (feature 3)
'''

means = torch.tensor([10, 20])
stds = torch.tensor([100, 200])
twos = torch.tensor([2, 2])

means_expanded = means.unsqueeze(1)
stds_expanded = stds.unsqueeze(1)
twos_expanded = twos.unsqueeze(1)

result = torch.cat((means_expanded, stds_expanded, twos_expanded), dim=1).unsqueeze(1)
print(result)

result_tensor = torch.cat((premises, result), dim=1)
print(result_tensor)

tensor([[[ 10, 100,   2]],

        [[ 20, 200,   2]]])
tensor([[[  1,   2,   3],
         [  4,   5,   6],
         [  7,   8,   9],
         [ 10, 100,   2]],

        [[  2,   4,   6],
         [  1,   3,   5],
         [  6,   8,  10],
         [ 20, 200,   2]]])


In [ ]:
twos = torch.ones_like(means).unsqueeze(1)*2
twos

tensor([[2],
        [2]])

In [ ]:
consequents = torch.tensor([[2, 4, 6],
                         [1, 3, 5],
                         [6, 8, 10]])
'''
        [[p1, q1, r1],  (consequent parameters of feature 1)
         [p2, q2, r2]]  (consequent parameters of feature 2)
         [p3, q3, r3]]  (consequent parameters of feature 3)
'''
new_consequent = torch.rand(3).unsqueeze(0)
print(new_consequent)

consequents = torch.cat((consequents, new_consequent))
print(consequents)

tensor([[0.3903, 0.6362, 0.5418]])
tensor([[ 2.0000,  4.0000,  6.0000],
        [ 1.0000,  3.0000,  5.0000],
        [ 6.0000,  8.0000, 10.0000],
        [ 0.3903,  0.6362,  0.5418]])


In [ ]:
#Ngrow and EpsilonGrow parameters are established by the user
def GrowNet(ANFISmodel, loader, Ngrow, dGrow):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Boolean mask to extract the necessary info
        mask = (max_fl.values <= dGrow**ANFISmodel.input_size)

        #Necesary tensors and bad samples dictionary are defined on the first iteration
        if first_batch:
            batch_bad_samples = torch.tensor([])
            best_bbs_rules = torch.tensor([]) #bbs -> batch bad samples
            bad_samples = {}
            for i in range(ANFISmodel.rules):
                bad_samples[i] = torch.tensor([])
            first_batch = False

        #The info is extracted by concatenating tensors
        batch_bad_samples = torch.cat((batch_bad_samples, x_batch[mask]), dim=0)
        best_bbs_rules = torch.cat((best_bbs_rules, max_fl.indices[mask]), dim=0)

        #The info is registered on the dictionary
        for index, rule in enumerate(best_bbs_rules):
            sample = batch_bad_samples[index, :].unsqueeze(0)
            bad_samples[rule.item()] = torch.cat((bad_samples[rule.item()], sample), dim=0)

    #Ngrow parameter filter
    bad_samples = {rule: samples for rule, samples in bad_samples.items() if samples.size(0) > Ngrow}

    #return False if ANFISmodel is not modified
    if bad_samples == {}:
        return False

    #Premises and consequents modifications to add a new rule
    for rule, samples in bad_samples.items():
        #mean and std are calculated with the samples of each obtained group
        means = samples.mean(dim=0).unsqueeze(1)
        stds = samples.std(dim=0).unsqueeze(1)

        premises = ANFISmodel.premises
        consequents = ANFISmodel.consequents

        #new premises depends of how many parameters uses the MFs
        if len(ANFISmodel.mf_params) == 2:
            new_premise = torch.cat([means, stds]).unsqueeze(1)
        elif len(ANFISmodel.mf_params) == 3:
            twos = torch.ones_like(means).unsqueeze(1)*2
            new_premise = torch.cat([means, stds, twos]).unsqueeze(1)

        #new consequents depends of how many dimentions has the input data
        new_consequent = torch.rand(ANFISmodel.input_size+1).unsqueeze(0)

        #The new rule is added
        '''
        It may be necessary using some torch function like self.premises.requires_grad = True
        to modify the premises
        '''
        ANFISmodel.set_premises(torch.cat((premises, new_premise), dim=1))
        ANFISmodel.set_consequents(torch.cat((consequents, new_consequent)))
        ANFISmodel.add_rule

    #return True if ANFISmodel is modified
    return True


###Proposal 2
Only with tensor operations

In [ ]:
x = torch.tensor([[1, 2], [2, 3], [3, 4], [4, 5], [5, 6], [6, 7], [7, 8], [8, 9]])

w = torch.tensor([[6, 5, 8], [2, 6, 5], [7, 7, 5], [9, 1, 2], [3, 2, 1], [5, 8, 6], [4, 2, 9], [2, 8, 5]])
''' wi = feature [i] firing level
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
w

tensor([[6, 5, 8],
        [2, 6, 5],
        [7, 7, 5],
        [9, 1, 2],
        [3, 2, 1],
        [5, 8, 6],
        [4, 2, 9],
        [2, 8, 5]])

In [ ]:
max_fl = torch.max(w, dim=1)
max_fl

torch.return_types.max(
values=tensor([8, 6, 7, 9, 3, 8, 9, 8]),
indices=tensor([2, 1, 0, 0, 0, 1, 2, 1]))

In [ ]:
mask = (max_fl.values <= 8)

In [ ]:
batch_bad_samples = torch.tensor([])
batch_bad_samples = torch.cat((batch_bad_samples, x[mask]), dim=0)
batch_bad_samples

tensor([[1., 2.],
        [2., 3.],
        [3., 4.],
        [5., 6.],
        [6., 7.],
        [8., 9.]])

In [ ]:
best_bbs_rules = torch.tensor([])
best_bbs_rules = torch.cat((best_bbs_rules, max_fl.indices[mask]), dim=0)
best_bbs_rules

tensor([2., 1., 0., 0., 1., 1.])

In [ ]:
unique_rules, counts = torch.unique(best_bbs_rules, return_counts=True)
unique_rules

tensor([0., 1., 2.])

In [ ]:
counts

tensor([2, 3, 1])

In [ ]:
mask = (counts > 1)

In [ ]:
counts[mask]

tensor([2, 3])

In [ ]:
unique_rules[mask]

tensor([0., 1.])

In [ ]:
unique_rules

tensor([0., 1., 2.])

In [ ]:
indices_to_keep = torch.isin(best_bbs_rules, unique_rules[mask]).nonzero().squeeze()

In [ ]:
indices_to_keep

tensor([1, 2, 3, 4, 5])

In [ ]:
batch_bad_samples = batch_bad_samples[indices_to_keep]
best_bbs_rules = best_bbs_rules[indices_to_keep]

In [ ]:
batch_bad_samples

tensor([[2., 3.],
        [3., 4.],
        [5., 6.],
        [6., 7.],
        [8., 9.]])

In [ ]:
best_bbs_rules

tensor([1., 0., 0., 1., 1.])

In [ ]:
batch_bad_samples.size(0)

5

In [ ]:
x = torch.tensor([[8., 9.],
                  [6., 7.],
                  [2., 3.],
                  [1., 2.],
                  [4., 5.],
                  [4., 6.],
                  [4., 5.]])
r = torch.tensor([1., 1., 1., 2., 3., 3., 2.])

masks = [r == value for value in torch.unique(r)]
print(masks)

x_means = torch.stack([(x[mask].mean(dim=0)) for mask in masks])
x_stds = torch.stack([(x[mask].std(dim=0)) for mask in masks])

print("x_means:", x_means)
print("x_stds:", x_stds)

[tensor([ True,  True,  True, False, False, False, False]), tensor([False, False, False,  True, False, False,  True]), tensor([False, False, False, False,  True,  True, False])]
x_means: tensor([[5.3333, 6.3333],
        [2.5000, 3.5000],
        [4.0000, 5.5000]])
x_stds: tensor([[3.0551, 3.0551],
        [2.1213, 2.1213],
        [0.0000, 0.7071]])


In [ ]:
twos = torch.ones_like(x_means)*2

In [ ]:
new_prem = torch.stack([x_means.t(), x_stds.t()], dim=2)
new_prem

tensor([[[5.3333, 3.0551],
         [2.5000, 2.1213],
         [4.0000, 0.0000]],

        [[6.3333, 3.0551],
         [3.5000, 2.1213],
         [5.5000, 0.7071]]])

In [ ]:
new_prem2 = torch.stack([x_means.t(), x_stds.t(), twos.t()], dim=2)
new_prem2

tensor([[[5.3333, 3.0551, 2.0000],
         [2.5000, 2.1213, 2.0000],
         [4.0000, 0.0000, 2.0000]],

        [[6.3333, 3.0551, 2.0000],
         [3.5000, 2.1213, 2.0000],
         [5.5000, 0.7071, 2.0000]]])

In [ ]:
'''
x1 premises
[[[mu, sigma],   (feature 1)
  [mu, sigma],   (feature 2)
  [mu, sigma]],  (feature 3)

x2 premises
 [[mu, sigma],   (feature 1)
  [mu, sigma],   (feature 2)
  [mu, sigma]]]  (feature 3)
'''
premises = torch.tensor([[[1, 2],
         [2, 3],
         [3, 4]],

        [[2, 4],
         [4, 6],
         [6, 8]]])

In [ ]:
update_prem = torch.cat([premises, new_prem], dim=1)
update_prem

tensor([[[1.0000, 2.0000],
         [2.0000, 3.0000],
         [3.0000, 4.0000],
         [5.3333, 3.0551],
         [2.5000, 2.1213],
         [4.0000, 0.0000]],

        [[2.0000, 4.0000],
         [4.0000, 6.0000],
         [6.0000, 8.0000],
         [6.3333, 3.0551],
         [3.5000, 2.1213],
         [5.5000, 0.7071]]])

In [ ]:
new_prem.size(1)

3

In [ ]:
consequents = torch.rand(5, 2+1)
consequents

tensor([[0.1115, 0.1754, 0.1625],
        [0.3175, 0.5911, 0.5846],
        [0.0627, 0.3515, 0.5720],
        [0.6872, 0.0591, 0.5270],
        [0.3982, 0.8436, 0.8473]])

In [ ]:
new_consequent = torch.rand(new_prem.size(1), 2+1)
new_consequent

tensor([[0.2867, 0.9764, 0.2459],
        [0.9532, 0.0107, 0.5759],
        [0.6769, 0.3201, 0.1678]])

In [ ]:
update_cons = torch.cat([consequents, new_consequent], dim=0)
update_cons

tensor([[0.1115, 0.1754, 0.1625],
        [0.3175, 0.5911, 0.5846],
        [0.0627, 0.3515, 0.5720],
        [0.6872, 0.0591, 0.5270],
        [0.3982, 0.8436, 0.8473],
        [0.2867, 0.9764, 0.2459],
        [0.9532, 0.0107, 0.5759],
        [0.6769, 0.3201, 0.1678]])

In [ ]:
def GrowNet(ANFISmodel, loader, Ngrow, dGrow):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Boolean mask to filter the samples
        dGrow_mask = (max_fl.values <= dGrow**ANFISmodel.input_size)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            bad_samples = torch.tensor([])
            best_bs_rules = torch.tensor([])
            first_batch = False

        #The samples are extracted by concatenating tensors (which are filtered by the mask)
        bad_samples = torch.cat((bad_samples, x_batch[dGrow_mask]), dim=0)
        best_bs_rules = torch.cat((best_bs_rules, max_fl.indices[dGrow_mask]), dim=0)

    #Ngrow parameter filter
    unique_rules, counts = torch.unique(best_bs_rules, return_counts=True)
    Ngrow_mask = (counts > Ngrow)

    indices_to_keep = torch.isin(best_bs_rules, unique_rules[Ngrow_mask]).nonzero().squeeze()

    bad_samples = bad_samples[indices_to_keep]
    best_bs_rules = best_bs_rules[indices_to_keep]

    #return False if ANFISmodel is not modified
    if bad_samples.size(0) == 0:
        return False

    #a list of masks called "rules" is created to calculate the necessary means and stds
    rules = [best_bs_rules == value for value in torch.unique(best_bs_rules)]

    #means and stds by rule are calculated
    means = torch.stack([(bad_samples[rule].mean(dim=0)) for rule in rules])
    stds = torch.stack([(bad_samples[rule].std(dim=0)) for rule in rules])

    #Premises and consequents modifications to add a new rule
    new_premises = torch.stack([means.t(), stds.t()], dim=2)
    ANFISmodel.set_premises(Parameter(torch.cat([ANFISmodel.premises.data, new_premises], dim=1), requires_grad=True))

    new_consequents = torch.rand(new_premises.size(1), ANFISmodel.input_size + 1)
    ANFISmodel.set_consequents(Parameter(torch.cat([ANFISmodel.consequents.data, new_consequents]), requires_grad=True))

    #return True if ANFISmodel is modified
    return True

####Testing

In [ ]:
x = torch.tensor([[5., 5.],
                  [6., 5.],
                  [7., 3.],
                  [1., 3.],
                  [4., 5.],
                  [4., 6.],
                  [4., 4.],
                  [3., 3.],
                  [2., 2.],
                  [1., 8.],
                  [2., 7.],
                  [8., 2.],
                  [9., 4.]])
y = torch.tensor([5., 6., 7., 3., 5., 6., 4., 3., 2., 8., 7., 8., 9.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 4)

In [ ]:
test_model = Type3ANFIS(x.shape[1], x_train = x)

In [ ]:
test_model.premises

Parameter containing:
tensor([[[1.0000, 2.0000],
         [5.0000, 2.0000],
         [9.0000, 2.0000]],

        [[2.0000, 1.5000],
         [5.0000, 1.5000],
         [8.0000, 1.5000]]], requires_grad=True)

In [ ]:
test_model.consequents

Parameter containing:
tensor([[0.2573, 0.7271, 0.7257],
        [0.7542, 0.9803, 0.9511],
        [0.5971, 0.3753, 0.1019]], requires_grad=True)

In [ ]:
test_model.firing_levels(x)

tensor([[5.4598e+01, 1.0000e+00, 5.4598e+01],
        [1.6817e+02, 1.1331e+00, 2.2760e+01],
        [1.1242e+02, 4.0104e+00, 4.2648e+02],
        [1.2488e+00, 1.7973e+01, 7.7109e+05],
        [2.2760e+01, 1.1331e+00, 1.6817e+02],
        [1.0783e+02, 1.4151e+00, 5.5362e+01],
        [7.4924e+00, 1.4151e+00, 7.9676e+02],
        [2.0590e+00, 4.0104e+00, 2.3285e+04],
        [1.1331e+00, 2.2760e+01, 1.3627e+06],
        [2.9810e+03, 5.4598e+01, 2.9810e+03],
        [2.9311e+02, 7.4924e+00, 5.7090e+02],
        [4.5714e+02, 2.2760e+01, 3.3779e+03],
        [7.2510e+03, 9.2278e+00, 3.5007e+01]], grad_fn=<ProdBackward1>)

In [ ]:
first_batch = True
for x_batch, y_batch in test_loader:
    #Max firing levels are obtenined
    firing_levels = test_model.firing_levels(x_batch)
    max_fl = torch.max(firing_levels, dim=1)

    #Boolean mask to filter the samples
    dGrow_mask = (max_fl.values <= 8.8*10**6)

    #Necesary tensors are defined on the first iteration
    if first_batch:
        bad_samples = torch.tensor([])
        best_bs_rules = torch.tensor([])
        first_batch = False

    #The samples are extracted by concatenating tensors (which are filtered by the mask)
    bad_samples = torch.cat((bad_samples, x_batch[dGrow_mask]), dim=0)
    best_bs_rules = torch.cat((best_bs_rules, max_fl.indices[dGrow_mask]), dim=0)

In [ ]:
bad_samples

tensor([[5., 5.],
        [6., 5.],
        [7., 3.],
        [1., 3.],
        [4., 5.],
        [4., 6.],
        [4., 4.],
        [3., 3.],
        [2., 2.],
        [1., 8.],
        [2., 7.],
        [8., 2.],
        [9., 4.]])

In [ ]:
best_bs_rules

tensor([0., 0., 2., 2., 2., 0., 2., 2., 2., 0., 2., 2., 0.])

In [ ]:
#Ngrow parameter filter
unique_rules, counts = torch.unique(best_bs_rules, return_counts=True)
Ngrow_mask = (counts > 2)

indices_to_keep = torch.isin(best_bs_rules, unique_rules[Ngrow_mask]).nonzero().squeeze()

bad_samples = bad_samples[indices_to_keep]
best_bs_rules = best_bs_rules[indices_to_keep]

In [ ]:
bad_samples

tensor([[5., 5.],
        [6., 5.],
        [7., 3.],
        [1., 3.],
        [4., 5.],
        [4., 6.],
        [4., 4.],
        [3., 3.],
        [2., 2.],
        [1., 8.],
        [2., 7.],
        [8., 2.],
        [9., 4.]])

In [ ]:
best_bs_rules

tensor([0., 0., 2., 2., 2., 0., 2., 2., 2., 0., 2., 2., 0.])

In [ ]:
#a list of masks called "rules" is created to calculate the necessary means and stds
rules = [best_bs_rules == value for value in torch.unique(best_bs_rules)]

#means and stds by rule are calculated
means = torch.stack([(bad_samples[rule].mean(dim=0)) for rule in rules])
stds = torch.stack([(bad_samples[rule].std(dim=0)) for rule in rules])

In [ ]:
means

tensor([[5.0000, 5.6000],
        [3.8750, 3.6250]])

In [ ]:
stds

tensor([[2.9155, 1.5166],
        [2.4749, 1.6850]])

In [ ]:
#Premises and consequents modifications to add a new rule
new_premises = torch.stack([means.t(), stds.t()], dim=2)
new_consequents = torch.rand(new_premises.size(1), test_model.input_size + 1)

In [ ]:
new_premises

tensor([[[5.0000, 2.9155],
         [3.8750, 2.4749]],

        [[5.6000, 1.5166],
         [3.6250, 1.6850]]])

In [ ]:
new_consequents

tensor([[0.2533, 0.3782, 0.0970],
        [0.8315, 0.3168, 0.4322]])

In [ ]:
test_model.set_premises(Parameter(torch.cat([test_model.premises, new_premises], dim=1), requires_grad=True))
test_model.set_consequents(Parameter(torch.cat([test_model.consequents, new_consequents]), requires_grad=True))

In [ ]:
test_model.premises

Parameter containing:
tensor([[[1.0000, 2.0000],
         [5.0000, 2.0000],
         [9.0000, 2.0000],
         [5.0000, 2.9155],
         [3.8750, 2.4749]],

        [[2.0000, 1.5000],
         [5.0000, 1.5000],
         [8.0000, 1.5000],
         [5.6000, 1.5166],
         [3.6250, 1.6850]]], requires_grad=True)

In [ ]:
test_model.consequents

Parameter containing:
tensor([[0.2573, 0.7271, 0.7257],
        [0.7542, 0.9803, 0.9511],
        [0.5971, 0.3753, 0.1019],
        [0.2533, 0.3782, 0.0970],
        [0.8315, 0.3168, 0.4322]], requires_grad=True)

###Proposal 3
Considering the ages_tensor for Vanish Operator

In [ ]:
x = torch.tensor([[7., 3.],
                  [1., 3.],
                  [4., 5.],
                  [4., 6.],
                  [4., 4.],
                  [3., 3.],
                  [2., 2.],
                  [1., 8.],
                  [2., 7.],
                  [8., 2.],
                  [9., 4.]])
y = torch.tensor([7., 3., 5., 6., 4., 3., 2., 8., 7., 8., 9.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 4)

In [ ]:
test_model = Type3ANFIS(x.shape[1], init_rules=4, x_train=x)

In [ ]:
ages = torch.zeros(test_model.rules)
#print(ages)

first_batch = True
for x_batch, y_batch in test_loader:
    #Max firing levels are obtenined
    firing_levels = test_model.firing_levels(x_batch)
    max_fl = torch.max(firing_levels, dim=1)
    #print(max_fl)
    #print(torch.unique(max_fl.indices))

    #Boolean mask to filter the samples
    dGrow_mask = (max_fl.values <= 5*10**8)

    #Necesary tensors are defined on the first iteration
    if first_batch:
        bad_samples = torch.tensor([])
        best_bs_rules = torch.tensor([])
        do_model = torch.zeros(test_model.rules, dtype=torch.bool) #Rules that do model samples
        first_batch = False

    #The samples are extracted by concatenating tensors (which are filtered by the mask)
    bad_samples = torch.cat((bad_samples, x_batch[dGrow_mask]), dim=0)
    best_bs_rules = torch.cat((best_bs_rules, max_fl.indices[dGrow_mask]), dim=0)

    #Rules that have samples on their Vk
    do_model[torch.unique(max_fl.indices)] = True

#print(bad_samples)
#print(best_bs_rules)
#print(do_model)
#print(ages)
ages[~do_model] += 1
#print(ages)

#Ngrow parameter filter
unique_rules, counts = torch.unique(best_bs_rules, return_counts=True)
Ngrow_mask = (counts > 2)

indices_to_keep = torch.isin(best_bs_rules, unique_rules[Ngrow_mask]).nonzero().squeeze()

bad_samples = bad_samples[indices_to_keep]
best_bs_rules = best_bs_rules[indices_to_keep]

#return False if ANFISmodel is not modified
#if bad_samples.size(0) == 0:
#    return False, ages

#a list of masks called "rules" is created to calculate the necessary means and stds
rules = [best_bs_rules == value for value in torch.unique(best_bs_rules)]
print(rules)

#means and stds by rule are calculated
means = torch.stack([(bad_samples[rule].mean(dim=0)) for rule in rules])
stds = torch.stack([(bad_samples[rule].std(dim=0)) for rule in rules])
print(ages)
print(test_model.premises.data)
print(test_model.consequents.data)

#Premises and consequents modifications to add a new rule
new_premises = torch.stack([means.t(), stds.t()], dim=2)
test_model.set_premises(Parameter(torch.cat([test_model.premises.data, new_premises], dim=1), requires_grad=True))

new_consequents = torch.rand(new_premises.shape[1], test_model.input_size + 1)
test_model.set_consequents(Parameter(torch.cat([test_model.consequents.data, new_consequents]), requires_grad=True))

ages = torch.cat([ages, torch.zeros(new_premises.shape[1], dtype=torch.bool)])
#return True if ANFISmodel is modified
#return True, ages
print(ages)
print(test_model.premises.data)
print(test_model.consequents.data)

[tensor([False, False,  True, False,  True, False, False,  True]), tensor([ True,  True, False,  True, False,  True,  True, False])]
tensor([0., 1., 1., 0.])
tensor([[[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333]],

        [[2.0000, 1.0000],
         [4.0000, 1.0000],
         [6.0000, 1.0000],
         [8.0000, 1.0000]]])
tensor([[0.2368, 0.5374, 0.7909],
        [0.2459, 0.2545, 0.6913],
        [0.9093, 0.7363, 0.6307],
        [0.6564, 0.5575, 0.1319]])
tensor([0., 1., 1., 0., 0., 0.])
tensor([[[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333],
         [4.6667, 4.0415],
         [5.0000, 2.4495]],

        [[2.0000, 1.0000],
         [4.0000, 1.0000],
         [6.0000, 1.0000],
         [8.0000, 1.0000],
         [6.0000, 2.0000],
         [4.2000, 1.9235]]])
tensor([[0.2368, 0.5374, 0.7909],
        [0.2459, 0.2545, 0.6913],
        [0.9093, 0.7363, 0.6307],
        [0.6564, 0.557

In [ ]:
def GrowNet(ANFISmodel, loader, ages, Ngrow, dGrow):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Boolean mask to filter the samples
        dGrow_mask = (max_fl.values <= dGrow**ANFISmodel.input_size)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            bad_samples = torch.tensor([])
            best_bs_rules = torch.tensor([])
            first_batch = False

        #The samples are extracted by concatenating tensors (which are filtered by the mask)
        bad_samples = torch.cat((bad_samples, x_batch[dGrow_mask]), dim=0)
        best_bs_rules = torch.cat((best_bs_rules, max_fl.indices[dGrow_mask]), dim=0)

    #Ngrow parameter filter
    unique_rules, counts = torch.unique(best_bs_rules, return_counts=True)
    Ngrow_mask = (counts > Ngrow)

    indices_to_keep = torch.isin(best_bs_rules, unique_rules[Ngrow_mask]).nonzero().squeeze()

    bad_samples = bad_samples[indices_to_keep]
    best_bs_rules = best_bs_rules[indices_to_keep]

    #return False if ANFISmodel is not modified
    if bad_samples.size(0) == 0:
        return False, ages

    #a list of masks called "rules" is created to calculate the necessary means and stds
    rules = [best_bs_rules == value for value in torch.unique(best_bs_rules)]

    #means and stds by rule are calculated
    means = torch.stack([(bad_samples[rule].mean(dim=0)) for rule in rules])
    stds = torch.stack([(bad_samples[rule].std(dim=0)) for rule in rules])

    #Premises and consequents modifications to add a new rule
    new_premises = torch.stack([means.t(), stds.t()], dim=2)
    ANFISmodel.set_premises(Parameter(torch.cat([ANFISmodel.premises.data, new_premises], dim=1), requires_grad=True))

    new_consequents = torch.rand(new_premises.size(1), ANFISmodel.input_size + 1)
    ANFISmodel.set_consequents(Parameter(torch.cat([ANFISmodel.consequents.data, new_consequents]), requires_grad=True))

    #New rules age
    ages = torch.cat([ages, torch.zeros(new_premises.shape[1], dtype=torch.bool)])

    #return True if ANFISmodel is modified
    return True, ages

####True Testing

##Split Sub-network

###Proposal 1

In [ ]:
def SplitSubNetwork(ANFISmodel, loader, Nsplit, eSplit):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Necesary tensors and best matching criteria dictionary are defined on the first iteration
        if first_batch:
            samples = torch.tensor([])
            rules = torch.tensor([])
            best_matching_criteria = {}
            for i in range(ANFISmodel.rules):
                best_matching_criteria[i] = torch.tensor([])
            first_batch = False

        #The info is extracted by concatenating tensors
        samples = torch.cat((samples, x_batch), dim=0)
        rules = torch.cat((rules, max_fl.indices), dim=0)

        #The info its registered on the dictionary
        for index, rule in enumerate(rules):
            sample = samples[index, :].unsqueeze(0)
            best_matching_criteria[rule.item()] = torch.cat((best_matching_criteria[rule.item()], sample), dim=0)

    #Nsplit parameter filter
    best_matching_criteria = {rule: samples for rule, samples in best_matching_criteria.items() if samples.size(0) > Nsplit}

    #eSplit parameter filter
    '''
    some
    thing
    '''

    #return False if ANFISmodel is not modified
    if best_matching_criteria == {}:
        return False

    #Premises and consequents modifications to split and existing rule into 2 new ones
    '''
    some
    thing
    '''

    #return True if ANFISmodel is modified
    return True

###Proposal 2
Only tensor operations

In [ ]:
x = torch.tensor([[7., 3.],
                  [1., 3.],
                  [4., 5.],
                  [4., 6.],
                  [4., 4.],
                  [3., 3.],
                  [2., 2.],
                  [1., 8.],
                  [2., 7.],
                  [8., 2.],
                  [9., 4.]])
y = torch.tensor([7., 3., 5., 6., 4., 3., 2., 8., 7., 8., 9.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 4)

In [ ]:
test_model = Type3ANFIS(x.shape[1], x_train = x)

In [ ]:
test_model.premises

Parameter containing:
tensor([[[1.0000, 2.0000],
         [5.0000, 2.0000],
         [9.0000, 2.0000]],

        [[2.0000, 1.5000],
         [5.0000, 1.5000],
         [8.0000, 1.5000]]], requires_grad=True)

In [ ]:
test_model.consequents

Parameter containing:
tensor([[0.3462, 0.0905, 0.0206],
        [0.3964, 0.4850, 0.9301],
        [0.1167, 0.9023, 0.3944]], requires_grad=True)

In [ ]:
test_model.firing_levels(x)

tensor([[1.1242e+02, 4.0104e+00, 4.2648e+02],
        [1.2488e+00, 1.7973e+01, 7.7109e+05],
        [2.2760e+01, 1.1331e+00, 1.6817e+02],
        [1.0783e+02, 1.4151e+00, 5.5362e+01],
        [7.4924e+00, 1.4151e+00, 7.9676e+02],
        [2.0590e+00, 4.0104e+00, 2.3285e+04],
        [1.1331e+00, 2.2760e+01, 1.3627e+06],
        [2.9810e+03, 5.4598e+01, 2.9810e+03],
        [2.9311e+02, 7.4924e+00, 5.7090e+02],
        [4.5714e+02, 2.2760e+01, 3.3779e+03],
        [7.2510e+03, 9.2278e+00, 3.5007e+01]], grad_fn=<ProdBackward1>)

In [ ]:
first_batch = True
for x_batch, y_batch in test_loader:
    #Max firing levels are obtenined
    firing_levels = test_model.firing_levels(x_batch)
    max_fl = torch.max(firing_levels, dim=1)

    #Necesary tensors are defined on the first iteration
    if first_batch:
        samples = torch.tensor([])
        samples_output = torch.tensor([])
        best_rules = torch.tensor([])
        first_batch = False

    #The samples and its best rule are extracted by concatenating tensors
    samples = torch.cat((samples, x_batch), dim=0)
    samples_output = torch.cat((samples_output, y_batch), dim=0)
    best_rules = torch.cat((best_rules, max_fl.indices), dim=0)

In [ ]:
best_rules

tensor([2., 2., 2., 0., 2., 2., 2., 0., 2., 2., 0.])

In [ ]:
#Nsplit parameter filter
unique_rules, counts = torch.unique(best_rules, return_counts=True)
Nsplit_mask = (counts > 2)

indices_to_keep = torch.isin(best_rules, unique_rules[Nsplit_mask]).nonzero().squeeze()

samples = samples[indices_to_keep]
samples_output = samples_output[indices_to_keep]
best_rules = best_rules[indices_to_keep]

In [ ]:
samples

tensor([[7., 3.],
        [1., 3.],
        [4., 5.],
        [4., 6.],
        [4., 4.],
        [3., 3.],
        [2., 2.],
        [1., 8.],
        [2., 7.],
        [8., 2.],
        [9., 4.]])

In [ ]:
best_rules

tensor([2., 2., 2., 0., 2., 2., 2., 0., 2., 2., 0.])

In [ ]:
unique_rules = torch.unique(best_rules)

In [ ]:
unique_rules

tensor([0., 2.])

In [ ]:
#MSE is calculated by rule
mse_values = torch.stack([torch.pow((samples_output[best_rules == rule] - test_model(samples[best_rules == rule])), 2).mean(dim=0) for rule in unique_rules])

In [ ]:
mse_values #all ready corroborated

tensor([16.5231,  4.9077], grad_fn=<StackBackward0>)

In [ ]:
#eSplit parameter filter
eSplit_mask = (mse_values > 5)

In [ ]:
unique_rules[eSplit_mask]

tensor([0.])

In [ ]:
torch.flip(unique_rules[eSplit_mask], [0])

tensor([0.])

In [ ]:
current_premises = test_model.premises.clone()
current_premises

tensor([[[1.0000, 2.0000],
         [5.0000, 2.0000],
         [9.0000, 2.0000]],

        [[2.0000, 1.5000],
         [5.0000, 1.5000],
         [8.0000, 1.5000]]], grad_fn=<CloneBackward0>)

In [ ]:
consequents = test_model.consequents.clone()
consequents

tensor([[0.3462, 0.0905, 0.0206],
        [0.3964, 0.4850, 0.9301],
        [0.1167, 0.9023, 0.3944]], grad_fn=<CloneBackward0>)

In [ ]:
t = torch.tensor([[[9.0000, 2.0000]],
                  [[8.0000, 1.5000]]])

print((t[:,:,0] + t[:,:,1]/2).unsqueeze(1))
print((t[:,:,1]/2).unsqueeze(1))
split1 = torch.cat([(t[:,:,0] + t[:,:,1]/2).unsqueeze(1), (t[:,:,1]/2).unsqueeze(1)], dim=2)
split2 = torch.cat([(t[:,:,0] - t[:,:,1]/2).unsqueeze(1), (t[:,:,1]/2).unsqueeze(1)], dim=2)
print(torch.cat([split1, split2], dim=1))

tensor([[[10.0000]],

        [[ 8.7500]]])
tensor([[[1.0000]],

        [[0.7500]]])
tensor([[[10.0000,  1.0000],
         [ 8.0000,  1.0000]],

        [[ 8.7500,  0.7500],
         [ 7.2500,  0.7500]]])


In [ ]:
new_consequents = torch.cat([consequents[:1, :], consequents[1+1:, :]], dim=0)
new_consequents

tensor([[0.3462, 0.0905, 0.0206],
        [0.1167, 0.9023, 0.3944]], grad_fn=<CatBackward0>)

In [ ]:
rand = torch.rand(2,3)
rand

tensor([[0.9210, 0.1885, 0.3485],
        [0.8005, 0.9643, 0.4524]])

In [ ]:
new_consequents = torch.cat([new_consequents, rand], dim=0)
new_consequents

tensor([[0.3462, 0.0905, 0.0206],
        [0.1167, 0.9023, 0.3944],
        [0.9210, 0.1885, 0.3485],
        [0.8005, 0.9643, 0.4524]], grad_fn=<CatBackward0>)

In [ ]:
new_premises = test_model.premises
new_consequents = test_model.consequents
for rule in list(torch.flip(unique_rules[eSplit_mask], [0]).long()):
    new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
    to_split = test_model.premises[:, rule:rule+1, :].clone()
    split1 = torch.cat([(to_split[:,:,0] - to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)
    split2 = torch.cat([(to_split[:,:,0] + to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)
    new_premises = torch.cat([new_premises, torch.cat([split1, split2], dim=1)], dim=1)

    new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)
    new_consequents = torch.cat([new_consequents, torch.rand(2,3)], dim=0)

In [ ]:
new_premises

tensor([[[5.0000, 2.0000],
         [9.0000, 2.0000],
         [0.0000, 1.0000],
         [2.0000, 1.0000]],

        [[5.0000, 1.5000],
         [8.0000, 1.5000],
         [1.2500, 0.7500],
         [2.7500, 0.7500]]], grad_fn=<CatBackward0>)

In [ ]:
new_consequents

tensor([[0.3964, 0.4850, 0.9301],
        [0.1167, 0.9023, 0.3944],
        [0.5206, 0.4696, 0.5460],
        [0.2397, 0.0870, 0.5750]], grad_fn=<CatBackward0>)

In [ ]:
def SplitSubNetwork(ANFISmodel, loader, Nsplit, eSplit):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            samples = torch.tensor([])
            samples_output = torch.tensor([])
            best_rules = torch.tensor([])
            first_batch = False

        #The samples and its best rule are extracted by concatenating tensors
        samples = torch.cat((samples, x_batch), dim=0)
        samples_output = torch.cat((samples_output, y_batch), dim=0)
        best_rules = torch.cat((best_rules, max_fl.indices), dim=0)

    #Nsplit parameter filter
    unique_rules, counts = torch.unique(best_rules, return_counts=True)
    Nsplit_mask = (counts > Nsplit)

    indices_to_keep = torch.isin(best_rules, unique_rules[Nsplit_mask]).nonzero().squeeze()

    samples = samples[indices_to_keep]
    samples_output = samples_output[indices_to_keep]
    best_rules = best_rules[indices_to_keep]

    #return Flase if ANFISmodel is not modified
    if samples.size(0) == 0:
        return False

    #the rules from best_rules tensor are extracted
    unique_rules = torch.unique(best_rules)

    #MSE is calculated by rule
    mse_values = torch.stack([torch.pow((samples_output[best_rules == rule] - ANFISmodel(samples[best_rules == rule])), 2).mean(dim=0) for rule in unique_rules])

    #eSplit parameter filter
    eSplit_mask = (mse_values > eSplit)

    #loop to split each rule one by one and generate the new ones
    new_premises = ANFISmodel.premises.data #new_premises starts being a copy of the current premises
    new_consequents = ANFISmodel.consequents.data #same thing with consequents
    for rule in list(torch.flip(unique_rules[eSplit_mask], [0]).long()): #the iteration is performed on a list with the rules in descending order
        #the selected premise is extracted from the new_premises tensor and placed into the to_split tensor
        new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
        to_split = ANFISmodel.premises.data[:, rule:rule+1, :].clone()

        #the new ones are generated
        split1 = torch.cat([(to_split[:,:,0] - to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)
        split2 = torch.cat([(to_split[:,:,0] + to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)

        #both are inserted on the new premises tensor
        new_premises = torch.cat([new_premises, torch.cat([split1, split2], dim=1)], dim=1)

        #the corresponding consequent is taken away
        new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)

        #two new consequents are added
        new_consequents = torch.cat([new_consequents, torch.rand(2, ANFISmodel.input_size + 1)], dim=0)

    #after the loop, the new parameters are set
    ANFISmodel.set_premises(Parameter(new_premises, requires_grad=True))
    ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

    #return True if ANFISmodel is modified
    return True

###Proposal 3

In [ ]:
x = torch.tensor([[7., 3.],
                  [1., 3.],
                  [4., 5.],
                  [4., 6.],
                  [4., 4.],
                  [3., 3.],
                  [2., 2.],
                  [1., 8.],
                  [2., 7.],
                  [8., 2.],
                  [9., 4.]])
y = torch.tensor([7., 3., 5., 6., 4., 3., 2., 8., 7., 8., 9.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 4)

In [ ]:
test_model2 = test_model

In [ ]:
test_model2.premises

Parameter containing:
tensor([[[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333],
         [4.6667, 4.0415],
         [5.0000, 2.4495]],

        [[2.0000, 1.0000],
         [4.0000, 1.0000],
         [6.0000, 1.0000],
         [8.0000, 1.0000],
         [6.0000, 2.0000],
         [4.2000, 1.9235]]], requires_grad=True)

In [ ]:
ages

tensor([0., 1., 1., 0., 0., 0.])

In [ ]:
first_batch = True
for x_batch, y_batch in test_loader:
    #Max firing levels are obtenined
    firing_levels = test_model2.firing_levels(x_batch)
    max_fl = torch.max(firing_levels, dim=1)
    #print(max_fl)
    #print(torch.unique(max_fl.indices))

    if first_batch:
        samples = torch.tensor([])
        samples_output = torch.tensor([])
        best_rules = torch.tensor([])
        first_batch = False

    #The samples and its best rule are extracted by concatenating tensors
    samples = torch.cat((samples, x_batch), dim=0)
    samples_output = torch.cat((samples_output, y_batch), dim=0)
    best_rules = torch.cat((best_rules, max_fl.indices), dim=0)

#Nsplit parameter filter
unique_rules, counts = torch.unique(best_rules, return_counts=True)
Nsplit_mask = (counts > 2)

indices_to_keep = torch.isin(best_rules, unique_rules[Nsplit_mask]).nonzero().squeeze()

samples = samples[indices_to_keep]
samples_output = samples_output[indices_to_keep]
best_rules = best_rules[indices_to_keep]

#the rules from best_rules tensor are extracted
unique_rules = torch.unique(best_rules)
#print(unique_rules)

#hola
#MSE is calculated by rule
mse_values = torch.stack([torch.pow((samples_output[best_rules == rule] - test_model2(samples[best_rules == rule])), 2).mean(dim=0) for rule in unique_rules])


#eSplit parameter filter
eSplit_mask = (mse_values > 5)

#loop to split each rule one by one and generate the new ones
new_premises = test_model2.premises.data #new_premises starts being a copy of the current premises
new_consequents = test_model2.consequents.data #same thing with consequents
print(new_premises)
print(new_consequents)
print(ages)
for rule in list(torch.flip(unique_rules[eSplit_mask], [0]).long()): #the iteration is performed on a list with the rules in descending order
    #the selected premise is extracted from the new_premises tensor and placed into the to_split tensor
    new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
    to_split = test_model2.premises.data[:, rule:rule+1, :].clone()

    #the new ones are generated
    split1 = torch.cat([(to_split[:,:,0] - to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)
    split2 = torch.cat([(to_split[:,:,0] + to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)

    #both are inserted on the new premises tensor
    new_premises = torch.cat([new_premises, torch.cat([split1, split2], dim=1)], dim=1)

    #the corresponding consequent is taken away
    new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)

    #two new consequents are added
    new_consequents = torch.cat([new_consequents, torch.rand(2, test_model2.input_size + 1)], dim=0)

    #same with ages tensor
    ages = torch.cat([ages[:rule], ages[rule+1:]]) #the corresponding rule is taken away
    ages = torch.cat([ages, torch.zeros(2, dtype=torch.bool)]) #the new ones are added

    print("------------------------")
    print(new_premises)
    print(new_consequents)
    print(ages)

#after the loop, the new parameters are set
#test_model2.set_premises(Parameter(new_premises, requires_grad=True))
#test_model2.set_consequents(Parameter(new_consequents, requires_grad=True))

tensor([[[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333],
         [4.6667, 4.0415],
         [5.0000, 2.4495]],

        [[2.0000, 1.0000],
         [4.0000, 1.0000],
         [6.0000, 1.0000],
         [8.0000, 1.0000],
         [6.0000, 2.0000],
         [4.2000, 1.9235]]])
tensor([[0.2368, 0.5374, 0.7909],
        [0.2459, 0.2545, 0.6913],
        [0.9093, 0.7363, 0.6307],
        [0.6564, 0.5575, 0.1319],
        [0.2909, 0.8566, 0.1361],
        [0.6131, 0.2499, 0.9328]])
tensor([0., 1., 1., 0., 0., 0.])
------------------------
tensor([[[3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333],
         [4.6667, 4.0415],
         [5.0000, 2.4495],
         [0.3333, 0.6667],
         [1.6667, 0.6667]],

        [[4.0000, 1.0000],
         [6.0000, 1.0000],
         [8.0000, 1.0000],
         [6.0000, 2.0000],
         [4.2000, 1.9235],
         [1.5000, 0.5000],
         [2.5000, 0.5000]]])
tensor([[0.2459, 0.2545, 0.

In [ ]:
def SplitSubNetwork(ANFISmodel, loader, ages, Nsplit, eSplit):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            samples = torch.tensor([])
            samples_output = torch.tensor([])
            best_rules = torch.tensor([])
            first_batch = False

        #The samples and its best rule are extracted by concatenating tensors
        samples = torch.cat((samples, x_batch), dim=0)
        samples_output = torch.cat((samples_output, y_batch), dim=0)
        best_rules = torch.cat((best_rules, max_fl.indices), dim=0)

    #Nsplit parameter filter
    unique_rules, counts = torch.unique(best_rules, return_counts=True)
    Nsplit_mask = (counts > Nsplit)

    indices_to_keep = torch.isin(best_rules, unique_rules[Nsplit_mask]).nonzero().squeeze()

    samples = samples[indices_to_keep]
    samples_output = samples_output[indices_to_keep]
    best_rules = best_rules[indices_to_keep]

    #return Flase if ANFISmodel is not modified
    if samples.size(0) == 0:
        return False, ages

    #the rules from best_rules tensor are extracted
    unique_rules = torch.unique(best_rules)

    #MSE is calculated by rule
    mse_values = torch.stack([torch.pow((samples_output[best_rules == rule] - ANFISmodel(samples[best_rules == rule])), 2).mean(dim=0) for rule in unique_rules])

    #eSplit parameter filter
    eSplit_mask = (mse_values > eSplit)

    #loop to split each rule one by one and generate the new ones
    new_premises = ANFISmodel.premises.data #new_premises starts being a copy of the current premises
    new_consequents = ANFISmodel.consequents.data #same thing with consequents
    for rule in list(torch.flip(unique_rules[eSplit_mask], [0]).long()): #the iteration is performed on a list with the rules in descending order
        #the selected premise is extracted from the new_premises tensor and placed into the to_split tensor
        new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
        to_split = ANFISmodel.premises.data[:, rule:rule+1, :].clone()

        #the new ones are generated
        split1 = torch.cat([(to_split[:,:,0] - to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)
        split2 = torch.cat([(to_split[:,:,0] + to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)

        #both are inserted on the new premises tensor
        new_premises = torch.cat([new_premises, torch.cat([split1, split2], dim=1)], dim=1)

        #the corresponding consequent is taken away
        new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)

        #two new consequents are added
        new_consequents = torch.cat([new_consequents, torch.rand(2, ANFISmodel.input_size + 1)], dim=0)

        #same with ages tensor
        ages = torch.cat([ages[:rule], ages[rule+1:]]) #the corresponding rule is taken away
        ages = torch.cat([ages, torch.zeros(2, dtype=torch.bool)]) #the new ones are added

    #after the loop, the new parameters are set
    ANFISmodel.set_premises(Parameter(new_premises, requires_grad=True))
    ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

    #return True if ANFISmodel is modified
    return True, ages

####Testing

####True Testing

##Vanish Sub-network

###Proposal 1
Only tensor operations, sample by sample

In [ ]:
fl = test_model.firing_levels(torch.tensor([2., 3.]))
fl

tensor([1.4151e+00, 7.4924e+00, 1.1825e+05], grad_fn=<ProdBackward1>)

In [ ]:
firing_levels = torch.rand(4,3)
''' wi = feature [i] firing level
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
firing_levels

tensor([[0.2213, 0.1867, 0.0612],
        [0.9475, 0.8846, 0.3086],
        [0.8230, 0.2538, 0.4853],
        [0.6232, 0.9526, 0.5182]])

In [ ]:
max_fl = torch.max(firing_levels, dim=1)
max_fl.indices

tensor([0, 0, 0, 1])

In [ ]:
ages = torch.zeros([firing_levels.shape[1]])
ages

tensor([0., 0., 0.])

In [ ]:
lVanish = 4
Nvanish = 4
first_sample = True
for x_batch, y_batch in test_loader:
    for sample in x_batch:
        #Max firing level is obtenined
        firing_level = test_model.firing_levels(sample)
        max_fl = torch.max(firing_level, dim=0)

        #Necesary tensors are defined on the first iteration
        if first_sample:
            ages = torch.zeros_like(firing_level)
            modeled_samples = torch.zeros_like(firing_level)
            #     best_rules = torch.tensor([])
            first_sample = False

        ages += 1
        ages[max_fl.indices] = 0
        modeled_samples[max_fl.indices] += 1 #the sample is counted
        #     best_rules = torch.cat((best_rules, torch.tensor([max_fl.indices])))

        #filter
ages

tensor([ 0., 11.,  1.])

In [ ]:
modeled_samples

tensor([3., 0., 8.])

In [ ]:
to_eliminate = ((ages > lVanish) & (modeled_samples < Nvanish)).nonzero().flatten()
to_eliminate

tensor([1])

In [ ]:
def VanishSubNetwork(ANFISmodel, loader, ages_tensor, Nvanish, lVanish):
    model_modified = False
    first_sample = True
    for x_batch, y_batch in loader:
        for sample in x_batch:
            #Max firing level is obtenined
            firing_level = ANFISmodel.firing_levels(sample)
            max_fl = torch.max(firing_levels, dim=0)

            #Necesary tensors are defined on the first iteration
            if first_sample:
                modeled_samples = torch.zeros_like(firing_level) #tensor for the amount of samples best modeled by rule
                first_sample = False

            #the age of the rule that best modeled the sample is set to 0
            ages[max_fl.indices] = 0
            modeled_samples[max_fl.indices] += 1 #the sample is counted for the corresponding rule

            #Nvanish and lVanish filter to eliminate a rule
            #if age exceeds lVanish AND modeled_samples is less than Nvanish
            to_eliminate = ((ages > lVanish) & (modeled_samples < Nvanish)).nonzero().flatten()

    #return True if ANFISmodel is modified,
    return model_modified

###Proposal 2
Only tensor operations, batch by batch

In [ ]:
x = torch.tensor([[7., 3.],
                  [1., 3.],
                  [4., 5.],
                  [4., 6.],
                  [4., 4.],
                  [3., 3.],
                  [2., 2.],
                  [1., 8.],
                  [2., 7.],
                  [8., 2.],
                  [9., 4.]])
y = torch.tensor([7., 3., 5., 6., 4., 3., 2., 8., 7., 8., 9.])

In [ ]:
test_loader = data.DataLoader(data.TensorDataset(x, y), batch_size = 4)

In [ ]:
test_model3 = test_model2

In [ ]:
test_model3.premises

Parameter containing:
tensor([[[1.0000, 2.0000],
         [5.0000, 2.0000],
         [9.0000, 2.0000]],

        [[2.0000, 1.5000],
         [5.0000, 1.5000],
         [8.0000, 1.5000]]], requires_grad=True)

In [ ]:
test_model3.consequents

Parameter containing:
tensor([[0.3462, 0.0905, 0.0206],
        [0.3964, 0.4850, 0.9301],
        [0.1167, 0.9023, 0.3944]], requires_grad=True)

In [ ]:
test_model3.rules

3

In [ ]:
tensor_original = torch.tensor([0., 3., 0., 2., 4., 1.])

indices_mayores_a_2 = torch.where(tensor_original > 2)[0]

print(indices_mayores_a_2)

tensor([1, 4])


In [ ]:
u = torch.tensor([0, 1, 2, 3])
u_counts = torch.tensor([1, 1, 4, 4])
r = torch.tensor([0, 2, 4])

mask = torch.isin(r, u)
not_in_r = r[~mask] # | (u_counts[torch.searchsorted(u, r)].le(3))
in_r = r[mask]

print(torch.sort(torch.cat([in_r[u_counts[in_r] > 3], not_in_r]), descending=True).values)

tensor([4, 2])


In [ ]:
lVanish = 2
Nvanish = 2
ages = torch.tensor([1., 1., 0., 0., 0., 0.])
last_best_rules = torch.tensor([3., 3., 3., 0., 3., 3., 3., 0., 3., 3., 0.])
first_batch = True
for x_batch, y_batch in test_loader:
    firing_levels = test_model3.firing_levels(x_batch)
    max_fl = torch.max(firing_levels, dim=1)

    #Necesary tensors are defined on the first iteration
    if first_batch:
        best_rules = torch.tensor([], dtype=torch.int)
        first_batch = False
        do_model = torch.zeros(test_model3.rules, dtype=torch.bool) #Rules that do model samples

    best_rules = torch.cat((best_rules, max_fl.indices), dim=0)
    do_model[torch.unique(max_fl.indices)] = True

#Suposicion para testing
best_rules = torch.tensor([3., 0., 1., 0., 2., 0., 3., 0., 2., 3., 2.], dtype=torch.int)

unique_rules, counts = torch.unique(best_rules, return_counts=True)

#ages update
if torch.equal(best_rules, last_best_rules):
    ages += 1
    #print(ages)
else:
    #print(unique_rules)
    last_unique_rules, last_counts = torch.unique(last_best_rules, return_counts=True)
    #print(counts)
    #print("---------")

    new_modeled_rules = ~torch.isin(unique_rules, last_unique_rules) #mask

    improved_rules = unique_rules[new_modeled_rules | (~new_modeled_rules & (counts > last_counts[torch.searchsorted(last_unique_rules, unique_rules)]))]
    all_rules = torch.arange(test_model3.rules)
    not_improved_rules = all_rules[~torch.isin(all_rules, improved_rules)]

    #print(improved_rules)
    #print(not_improved_rules)
    #print(ages)

    ages[improved_rules] = 0
    ages[not_improved_rules] += 1

    #for testing
    ages = torch.tensor([3., 3., 3., 3., 3., 3.])
    #print(ages)
    #print("---------")

#lVanish filter
rules_to_eliminate = torch.where(ages > lVanish)[0]
print(rules_to_eliminate)

total_counts = torch.zeros(test_model3.rules, dtype=rules_to_eliminate.dtype)
print(total_counts)
print(total_counts.dtype)
print(unique_rules)
print(counts)
total_counts[unique_rules] = counts
print(total_counts)

#Nvanish filter
mask = torch.isin(rules_to_eliminate, unique_rules)
print(mask)
rules_to_eliminate = torch.sort(torch.cat([rules_to_eliminate[~mask][counts[rules_to_eliminate[~mask]] < Nvanish], rules_to_eliminate[mask]]), descending=True).values
#print("---------")
#print(rules_to_eliminate)

#Parameters update
new_premises = test_model3.premises.data
new_consequents = test_model3.consequents.data
print(new_premises)
print(new_consequents)
for rule in rules_to_eliminate:
    new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
    new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)
print(new_premises)
print(new_consequents)

tensor([0, 1, 2, 3, 4, 5])
tensor([0, 0, 0])
torch.int64
tensor([0, 1, 2, 3], dtype=torch.int32)
tensor([4, 1, 3, 3])


IndexError: index 3 is out of bounds for dimension 0 with size 3

In [ ]:
def VanishSubNetwork(ANFISmodel, loader, ages, Nvanish, lVanish, last_best_rules):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            best_rules = torch.tensor([])
            first_batch = False

        best_rules = torch.cat((best_rules, max_fl.indices), dim=0)

    if torch.equal(best_rules, last_best_rules): #if there is no changes with the modeled samples
        #ages update
        ages += 1
    else:
        #rule categorization
        unique_rules, counts = torch.unique(best_rules, return_counts=True)
        last_unique_rules, last_counts = torch.unique(last_best_rules, return_counts=True)

        new_modeled_rules = ~torch.isin(unique_rules, last_unique_rules) #mask

        improved_rules = unique_rules[new_modeled_rules | (~new_modeled_rules & (counts > last_counts[torch.searchsorted(last_unique_rules, unique_rules)]))]
        all_rules = torch.arange(ANFISmodel.rules)
        not_improved_rules = all_rules[~torch.isin(all_rules, improved_rules)]

        #ages update
        ages[improved_rules] = 0
        ages[not_improved_rules] += 1

        #lVanish and Nvanish filters
        rules_to_eliminate = torch.where(ages > lVanish)[0]
        mask = torch.isin(rules_to_eliminate, unique_rules)
        rules_to_eliminate = torch.sort(torch.cat([rules_to_eliminate[mask][counts[rules_to_eliminate[mask]] < Nvanish], rules_to_eliminate[~mask]]), descending=True).values

        #Parameters update (the rules are eliminated)
        new_premises = ANFISmodel.premises.data
        new_consequents = ANFISmodel.consequents.data
        for rule in rules_to_eliminate:
            new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
            new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)

        #New parameters are set
        ANFISmodel.set_premises(Parameter(new_premises, requires_grad=True))
        ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

    #The best rules tensor is returned for next iterations, also the ages
    return best_rules, ages

###Proposal 3

In [ ]:
test_model3 = test_model2

In [ ]:
test_model3.premises

Parameter containing:
tensor([[[1.0000, 1.3333],
         [3.6667, 1.3333],
         [6.3333, 1.3333],
         [9.0000, 1.3333],
         [4.6667, 4.0415],
         [5.0000, 2.4495]],

        [[2.0000, 1.0000],
         [4.0000, 1.0000],
         [6.0000, 1.0000],
         [8.0000, 1.0000],
         [6.0000, 2.0000],
         [4.2000, 1.9235]]], requires_grad=True)

In [ ]:
lVanish = 2
Nvanish = 2
ages = torch.tensor([1., 1., 0., 0., 0., 0.])
last_best_rules = torch.tensor([3., 3., 3., 0., 3., 3., 3., 0., 3., 3., 0.])

In [ ]:
first_batch = True
for x_batch, y_batch in test_loader:
    firing_levels = test_model3.firing_levels(x_batch)
    max_fl = torch.max(firing_levels, dim=1)

    #Necesary tensors are defined on the first iteration
    if first_batch:
        best_rules = torch.tensor([], dtype=torch.int)
        first_batch = False
        do_model = torch.zeros(test_model3.rules, dtype=torch.bool) #Rules that do model samples

    best_rules = torch.cat((best_rules, max_fl.indices), dim=0)
    do_model[torch.unique(max_fl.indices)] = True

#Suposicion para testing
best_rules = torch.tensor([3., 0., 1., 0., 2., 0., 3., 0., 2., 3., 2.], dtype=torch.int)

unique_rules, counts = torch.unique(best_rules, return_counts=True)
all_rules = torch.arange(test_model3.rules)

#ages update
if torch.equal(best_rules, last_best_rules) | torch.equal(last_best_rules, torch.tensor([-1])):
    #ages update
    ages += 1
else:
    #rule categorization
    last_unique_rules, last_counts = torch.unique(last_best_rules, return_counts=True)

    new_modeled_rules = ~torch.isin(unique_rules, last_unique_rules) #mask

    improved_rules = unique_rules[new_modeled_rules | (~new_modeled_rules & (counts > last_counts[torch.searchsorted(last_unique_rules, unique_rules)]))]
    not_improved_rules = all_rules[~torch.isin(all_rules, improved_rules)]

    #ages update
    ages[improved_rules] = 0
    ages[not_improved_rules] += 1

    #for testing
    ages = torch.tensor([0., 3., 0., 3., 1., 3.])
    #print(ages)
    #print("---------")

total_counts = torch.zeros(test_model3.rules)
print("total_counts.dtype:", total_counts.dtype)
print("counts.dtype:", counts.dtype)
print("unique_rules.dtype:", unique_rules.dtype)
total_counts[unique_rules] = counts
print("total_counts: ", total_counts)

#lVanish filter
print("ages:", ages)
print("ages > lVanish:", ages > lVanish)
print("total_counts", total_counts)
print("total_counts < Nvanish:", total_counts < Nvanish)
mask = ((ages > lVanish) & (total_counts < Nvanish))

rules_to_eliminate = all_rules[mask]

print("rules_to_eliminate: ", rules_to_eliminate)
#print("---------")
#print(rules_to_eliminate)
'''
#Parameters update
new_premises = test_model3.premises.data
new_consequents = test_model3.consequents.data
print(new_premises)
print(new_consequents)
for rule in rules_to_eliminate:
    new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
    new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)
print(new_premises)
print(new_consequents)
'''

total_counts.dtype: torch.float32
counts.dtype: torch.int64
unique_rules.dtype: torch.int32


RuntimeError: Index put requires the source and destination dtypes match, got Float for the destination and Long for the source.

In [ ]:
def VanishSubNetwork(ANFISmodel, loader, ages, Nvanish, lVanish, last_best_rules):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            best_rules = torch.tensor([], dtype=torch.int)
            first_batch = False

        best_rules = torch.cat((best_rules, max_fl.indices), dim=0)

    unique_rules, counts = torch.unique(best_rules, return_counts=True)

    #if there is no changes with the modeled samples or there is no last_best_rules (first iteration of VanishNet operator)
    if torch.equal(best_rules, last_best_rules) | torch.equal(last_best_rules, torch.tensor([-1])):
        #ages update
        ages += 1
    else:
        #rule categorization
        last_unique_rules, last_counts = torch.unique(last_best_rules, return_counts=True)

        new_modeled_rules = ~torch.isin(unique_rules, last_unique_rules) #mask

        improved_rules = unique_rules[new_modeled_rules | (~new_modeled_rules & (counts > last_counts[torch.searchsorted(last_unique_rules, unique_rules)]))]
        all_rules = torch.arange(ANFISmodel.rules)
        not_improved_rules = all_rules[~torch.isin(all_rules, improved_rules)]

        #ages update
        ages[improved_rules] = 0
        ages[not_improved_rules] += 1

    #lVanish and Nvanish filters
    rules_to_eliminate = torch.where(ages > lVanish)[0]
    mask = torch.isin(rules_to_eliminate, unique_rules)

    total_counts = torch.zeros(ANFISmodel.rules, dtype=rules_to_eliminate.dtype)
    total_counts[unique_rules] = counts

    '''
    print("rules_to_eliminate:", rules_to_eliminate)
    print("mask:", mask)
    print("counts:", counts)
    print("rules_to_eliminate[mask]:", rules_to_eliminate[mask])
    print("mask counts:", counts[rules_to_eliminate[mask]])
    print("Nvanish:", Nvanish)
    print("Final tensor size:", rules_to_eliminate.size())
    '''
    rules_to_eliminate = torch.sort(torch.cat([rules_to_eliminate[mask][total_counts[rules_to_eliminate[mask]] < Nvanish], rules_to_eliminate[~mask]]), descending=True).values

    #Parameters update (the rules are eliminated)
    new_premises = ANFISmodel.premises.data
    new_consequents = ANFISmodel.consequents.data
    for rule in rules_to_eliminate:
        new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
        new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)
        ages = torch.cat([ages[:rule], ages[rule+1:]]) #the corresponding rule is taken away

    #New parameters are set
    ANFISmodel.set_premises(Parameter(new_premises, requires_grad=True))
    ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

    #The best rules tensor is returned for next iterations, also the ages
    return True, best_rules, ages